（１）乗降客数

In [ ]:
!pip install jaconv
!pip install dbfread

In [ ]:
import requests
import json
import pandas as pd
import numpy as np
import re
import os
import glob
import geopandas as gpd
from shapely.geometry import Point, Polygon
from geopandas.tools import sjoin
import shapely.wkt
import zipfile
from shapely.ops import nearest_points
from sklearn.cluster import DBSCAN
import pyogrio
from sklearn.metrics.pairwise import haversine_distances
from geopy.distance import geodesic
import jaconv

In [ ]:
# Googleドライブのマウント
from google.colab import drive
drive.mount("/content/drive")
%cd "/content/drive/My Drive/オープンデータチャレンジ2024"

In [ ]:
# 列の表示幅を広げる
# pd.set_option('display.max_colwidth', None)

# 行数や列数を制限せずに表示
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# アクセストークン
access_token = 'hid49u64myobsb2buk6pcqdlbq6baut8slmdpx7vjecemq15dhrzao8qwr3brwg9'

# リクエストのエンドポイント
url = 'https://api-challenge2024.odpt.org/api/v4/odpt:PassengerSurvey'

# 各operatorをリストに格納
operators = [
    'odpt.Operator:JR-East', 'odpt.Operator:Tobu', 'odpt.Operator:Tokyu',
    'odpt.Operator:Seibu', 'odpt.Operator:Keikyu', 'odpt.Operator:Odakyu',
    'odpt.Operator:MIR', 'odpt.Operator:TWR', 'odpt.Operator:YokohamaMunicipal',
    'odpt.Operator:TamaMonorail', 'odpt.Operator:Toei', 'odpt.Operator:TokyoMetro',
    'odpt.Operator:Sotetsu','odpt.Operator:Keio'
]

# 結果を格納するリストを初期化
all_rows = []

# 各operatorについてデータを取得し、処理
for operator in operators:
    # クエリパラメータ
    params = {
        'odpt:operator': operator,
        'acl:consumerKey': access_token
    }

    # リクエストを送信
    response = requests.get(url, params=params)

    # レスポンスのステータスを確認
    if response.status_code == 200:
        json_data = response.json()  # JSON形式でレスポンスを取得
        print(f"Data retrieved successfully for {operator}")

        # 各PassengerSurveyオブジェクトを処理
        for survey in json_data:
            base_row = {
                'sameAs': survey['owl:sameAs'],
                'railway': ', '.join(survey['odpt:railway']),
                'station': ', '.join(survey['odpt:station']),
                'operator': survey['odpt:operator'],
                'includeAlighting': survey['odpt:includeAlighting']
            }

            # passengerSurveyObjectの各エントリを処理
            for obj in survey['odpt:passengerSurveyObject']:
                row = base_row.copy()
                row['surveyYear'] = obj['odpt:surveyYear']
                row['passengerJourneys'] = obj['odpt:passengerJourneys']
                all_rows.append(row)

    else:
        print(f"Error {response.status_code} for {operator}")

# DataFrameを作成
df = pd.DataFrame(all_rows)

# surveyYearをピボットして新しい列を作成
df_pivot = df.pivot(index=['sameAs', 'railway', 'station', 'operator', 'includeAlighting'],
                    columns='surveyYear',
                    values='passengerJourneys')

# 列名を変更
df_pivot.columns = [f'passengers_{year}' for year in df_pivot.columns]

# インデックスをリセット
df_passenger = df_pivot.reset_index()

# 結果を表示
df_passenger

In [ ]:
# : の右側を抽出
df_passenger['trans_sameAs'] = df_passenger['sameAs'].apply(lambda x: x.split(':')[-1])

# sameAs列の.と.の間の文字列を削除
df_passenger['trans_sameAs'] = df_passenger['trans_sameAs'].apply(lambda x: re.sub(r'\.[^.]+\.', '.', x))

In [ ]:
df_passenger

（２）駅情報：緯度・経度

In [ ]:
import requests
import pandas as pd

# アクセストークン
access_token = 'hid49u64myobsb2buk6pcqdlbq6baut8slmdpx7vjecemq15dhrzao8qwr3brwg9'

# リクエストのエンドポイント
url = 'https://api-challenge2024.odpt.org/api/v4/odpt:Station'

# 各operatorをリストに格納
operators = [
    'odpt.Operator:JR-East', 'odpt.Operator:Tobu', 'odpt.Operator:Tokyu',
    'odpt.Operator:Seibu', 'odpt.Operator:Keikyu', 'odpt.Operator:Odakyu',
    'odpt.Operator:MIR', 'odpt.Operator:TWR', 'odpt.Operator:YokohamaMunicipal',
    'odpt.Operator:TamaMonorail', 'odpt.Operator:Toei', 'odpt.Operator:TokyoMetro',
    'odpt.Operator:Sotetsu','odpt.Operator:Sotetsu','odpt.Operator:Keio'
]

# 結果を格納するリストを初期化
all_rows = []

# 各operatorについてデータを取得し、処理
for operator in operators:
    # クエリパラメータ
    params = {
        'odpt:operator': operator,
        'acl:consumerKey': access_token
    }

    # リクエストを送信
    response = requests.get(url, params=params)

    # レスポンスのステータスを確認
    if response.status_code == 200:
        json_data = response.json()  # JSON形式でレスポンスを取得
        print(f"Data retrieved successfully for {operator}")

        # 各Stationオブジェクトを処理
        for station in json_data:
            row = {
                'sameAs': station.get('owl:sameAs', None),
                'station': station.get('dc:title', None),
                'stationCode': station.get('odpt:stationCode', None),
                'operator': station.get('odpt:operator', None),
                'railway': station.get('odpt:railway', None),
                'latitude': station.get('geo:lat', None),
                'longitude': station.get('geo:long', None),
                'connectingRailway': ', '.join(station.get('odpt:connectingRailway', [])),
                'connectingStation': ', '.join(station.get('odpt:connectingStation', []))
            }
            all_rows.append(row)

    else:
        print(f"Error {response.status_code} for {operator}")

# DataFrameを作成
df = pd.DataFrame(all_rows)

# 結果を表示
df_station = df.reset_index(drop=True)

# 結果を表示
df_station

In [ ]:
# latitude列にNaNがある行を削除
df_station = df_station.dropna(subset=['latitude'])

In [ ]:
df_station

In [ ]:
# connectingRailway列のodptの数をカウントし、新しい列として追加
df_station['connectingRailway_number'] = df_station['connectingRailway'].apply(
    lambda x: len([item.strip() for item in x.split(',') if item.strip()]) if pd.notnull(x) and x.strip() != "" else 0
)

df_station['connectingStation_number'] = df_station['connectingStation'].apply(
    lambda x: len([item.strip() for item in x.split(',') if item.strip()]) if pd.notnull(x) and x.strip() != "" else 0
)

# 結果の表示
df_station

In [ ]:
# 変換関数の定義
def transform_station_name(station_name):
    # "odpt.Station:" を削除
    station_name = station_name.replace('odpt.Station:', '')
    # 最初のドット以降の部分（鉄道路線名）を削除
    parts = station_name.split('.')
    if len(parts) > 2:
        return f"{parts[0]}.{parts[-1]}"  # 鉄道会社名と駅名だけを返す
    return station_name

# sameAs列の変換
df_station['trans_sameAs'] = df_station['sameAs'].apply(transform_station_name)

df_station

In [ ]:
# グループごとにフィルタリングする関数
def select_representative(group):
    # latitudeとlongitudeの組み合わせでカウント
    count = group.groupby(['latitude', 'longitude']).size().reset_index(name='count')
    # 最大カウントを持つ組み合わせを取得
    max_count = count['count'].max()
    # 最大カウントの候補を抽出
    candidates = count[count['count'] == max_count]

    if len(candidates) > 1:
        # 複数候補がある場合、元のデータフレームでインデックスが最小のものを選択
        # 元のデータフレームから候補にマッチする行を抽出
        representative_candidates = group.merge(candidates[['latitude', 'longitude']], on=['latitude', 'longitude'])
        representative = representative_candidates.loc[representative_candidates.index.min()]
    else:
        # 1つしかない場合、そのまま選択
        representative = group.merge(candidates[['latitude', 'longitude']], on=['latitude', 'longitude']).iloc[0]

    return representative

# sameAsとstationでグループ化し、代表行を選択
df_station = df_station.groupby(['trans_sameAs','station'], as_index=False).apply(select_representative).reset_index(drop=True)

# 結果の表示
df_station

In [ ]:
# df_stationのstation列のカラム名をjan_stationに変更
df_station = df_station.rename(columns={'station': 'jan_station'})

# データフレームを横結合する
df_merged = df_passenger.merge(df_station[['trans_sameAs', 'jan_station', 'connectingRailway_number', 'connectingStation_number', 'latitude', 'longitude']],
                               on='trans_sameAs',
                               how='left')

df_merged

In [ ]:
# lat列がNaNの行のみを抽出
df_lat_nan = df_merged[df_merged['latitude'].isna()]

# 結果を表示
df_lat_nan

In [ ]:
# jan_station 列が "海老名" の行を表示
filtered_df = df_merged[df_merged['jan_station'] == '越生']
filtered_df

In [ ]:
# jan_station列の「市ヶ谷」（東京都交通局）を「市ケ谷」に修正
df_merged['jan_station'] = df_merged['jan_station'].str.replace('市ヶ谷', '市ケ谷', regex=False)

In [ ]:
filtered_df = df_merged[df_merged['jan_station'] == '市ケ谷']
filtered_df

（３）国土数値情報から乗降客数データを読み込み（全国へ拡大）

In [ ]:
# GeoJSONファイルのパスを指定して読み込む
geojson_file = 'passengers_national_data_2022/S12-23_NumberOfPassengers.geojson'
passanger_national_2022 = gpd.read_file(geojson_file)
passanger_national_2022

In [ ]:
# connectingRailway_numberをカウントして追加
# 'S12_001'と'S12_001g'でグループ化し、各グループの行数をカウントして1を引きます
passanger_national_2022['connectingRailway_number'] = passanger_national_2022.groupby(['S12_001', 'S12_001g'])['S12_001'].transform('count') - 1
passanger_national_2022

In [ ]:
# 'column_name' の列に "Tokyo" を含む行を抽出
df_tokyo = passanger_national_2022[passanger_national_2022['S12_001'].str.contains('東京', case=False, na=False)]
df_tokyo

In [ ]:
unique_values = passanger_national_2022['S12_002'].unique()
print(unique_values)

In [ ]:
# 削除したい文字列のリスト
strings_to_remove = ['東日本旅客鉄道', '東武鉄道', '東急電鉄', '西武鉄道', '京浜急行電鉄','小田急電鉄',
                     '首都圏新都市鉄道','東京臨海高速鉄道','横浜市','多摩都市モノレール','東京都','東京地下鉄',
                     '相模鉄道','京王電鉄']

# S12_002列に指定した文字列が含まれる行を削除
passanger_national_2022 = passanger_national_2022[~passanger_national_2022['S12_002'].isin(strings_to_remove)]

In [ ]:
# S12_006列が2の行（重複し他で乗降客数を換算している行）を削除
passanger_national_2022 = passanger_national_2022[passanger_national_2022['S12_050'] != 2]

In [ ]:
# # 'column_name' の列に "Tokyo" を含む行を抽出
# df_tokyo = passanger_national_2022[passanger_national_2022['jan_station'].str.contains('JA広島病院前', case=False, na=False)]
# df_tokyo

In [ ]:
# 中心点を計算する関数
def calculate_centroid(geometry):
    centroid = geometry.centroid  # すでにジオメトリオブジェクトなのでそのままcentroidを使用
    return pd.Series({'latitude': centroid.y, 'longitude': centroid.x})

# 各行に関数を適用
passanger_national_2022[['latitude', 'longitude']] = passanger_national_2022['geometry'].apply(calculate_centroid)

In [ ]:
# 変更するカラム名を辞書として定義
column_mapping = {'S12_001': 'jan_station', 'S12_002': 'operator','S12_003': 'railway',
                  'S12_009': 'passengers_2011','S12_013': 'passengers_2012','S12_017': 'passengers_2013',
                  'S12_021': 'passengers_2014','S12_025': 'passengers_2015','S12_029': 'passengers_2016',
                  'S12_033': 'passengers_2017','S12_037': 'passengers_2018','S12_041': 'passengers_2019',
                  'S12_045': 'passengers_2020','S12_049': 'passengers_2021','S12_053': 'passengers_2022'}

# 変数を使ってカラム名を変更
passanger_national_2022 = passanger_national_2022.rename(columns=column_mapping)

passanger_national_2022

In [ ]:
# # jan_station列で重複している行のみを抽出
# duplicate_rows = passanger_national_2022[passanger_national_2022['jan_station'].duplicated(keep=False)]
# # jan_station列でソート
# duplicate_rows = duplicate_rows.sort_values(by='jan_station')
# # インデックスを0からリセット
# duplicate_rows = duplicate_rows.reset_index(drop=True)
# duplicate_rows

In [ ]:
# 国土数値情報の羽田空港を修正
passanger_national_2022['jan_station'] = passanger_national_2022['jan_station'].replace('羽田空港第1ターミナル', '羽田空港第１・第２ターミナル')
passanger_national_2022['jan_station'] = passanger_national_2022['jan_station'].replace('羽田空港第2ターミナル', '羽田空港第１・第２ターミナル')
passanger_national_2022['jan_station'] = passanger_national_2022['jan_station'].replace('羽田空港第3ターミナル', '羽田空港第３ターミナル')
passanger_national_2022

In [ ]:
# 国土数値情報の諫早を修正
passanger_national_2022['jan_station'] = passanger_national_2022['jan_station'].replace('諫早（雲仙・島原口）', '諫早')

passanger_national_2022['jan_station'] = passanger_national_2022['jan_station'].replace('空港第2ビル', '空港第２ビル')

passanger_national_2022['jan_station'] = passanger_national_2022['jan_station'].replace('試験場前', '聖マリア病院前')

In [ ]:
passanger_national_2022

In [ ]:
# jan_station 列が "海老名" の行を表示
filtered_df = passanger_national_2022[passanger_national_2022['jan_station'] == '海老名']
filtered_df

In [ ]:
# passanger_national_2022 の列を df_merged の列に揃える
passanger_national_2022 = passanger_national_2022.reindex(columns=df_merged.columns)

# 縦結合 (共通の列以外は NaN が挿入される)
df_merged = pd.concat([df_merged, passanger_national_2022], ignore_index=True, sort=False)

df_merged

In [ ]:
filtered_df = df_merged[df_merged['jan_station'] == '青海']
filtered_df

In [ ]:
#半径1km以内、かつ、jan_stationが一致する駅をグループ化して乗客数を加算
# 地球の半径（キロメートル）
R = 6371.0088

# 乗客数の列名リストを変数として定義
passenger_columns = ['passengers_2000', 'passengers_2001', 'passengers_2002', 'passengers_2003', 'passengers_2004',
                     'passengers_2005', 'passengers_2006', 'passengers_2007', 'passengers_2008', 'passengers_2009',
                     'passengers_2010', 'passengers_2011', 'passengers_2012', 'passengers_2013', 'passengers_2014',
                     'passengers_2015', 'passengers_2016', 'passengers_2017', 'passengers_2018', 'passengers_2019',
                     'passengers_2020', 'passengers_2021', 'passengers_2022', 'passengers_2023']

# その他の列名リストを変数として定義
other_columns = ['sameAs', 'railway', 'station', 'operator', 'includeAlighting', 'trans_sameAs', 'jan_station',
                 'connectingRailway_number', 'connectingStation_number', 'latitude', 'longitude']

# 最終的な列の順番を定義
final_columns = ['sameAs', 'railway', 'station', 'operator', 'includeAlighting'] + passenger_columns + \
                ['trans_sameAs', 'jan_station', 'connectingRailway_number', 'connectingStation_number', 'latitude', 'longitude']

# jan_stationごとに処理を行う関数を定義
def cluster_and_aggregate(group):
    # 緯度と経度をラジアンに変換
    coords = np.radians(group[['latitude', 'longitude']].values)
    # 1kmをラジアンに変換
    epsilon = 1 / R
    # DBSCANクラスタリングを実行
    db = DBSCAN(eps=epsilon, min_samples=1, algorithm='ball_tree', metric='haversine')
    cluster_labels = db.fit_predict(coords)
    group = group.copy()
    group['cluster'] = cluster_labels
    # クラスタごとに集計
    def agg_func(subgroup):
        # クラスタ内の行数を確認
        if len(subgroup) == 1:
            # 乗客数の列は元の値を保持
            passengers = subgroup[passenger_columns].iloc[0]
        else:
            # 乗客数の列を合計
            passengers = subgroup[passenger_columns].sum()
        # インデックスが最小の行を取得
        min_index_row = subgroup.loc[subgroup['index'].idxmin()]
        # その他の列を取得
        other_data = min_index_row[other_columns]
        # 必要な列を結合し、列の順番を指定
        result = pd.concat([other_data, passengers], axis=0)
        # 最終的な列の順番に合わせて並べ替え
        result = result[final_columns]
        return result
    # 修正ポイント：グループ化した列を明示的に含める
    aggregated = group.groupby('cluster', group_keys=False)[group.columns].apply(agg_func).reset_index(drop=True)
    return aggregated

# データフレームのインデックスを列としてリセット
df_merged = df_merged.reset_index()

# jan_stationごとに関数を適用
df_merged = df_merged.groupby('jan_station', as_index=False).apply(cluster_and_aggregate).reset_index(drop=True)

# 最終的なデータフレームの列順を指定（念のため）
df_merged = df_merged[final_columns]

# 結果を表示
df_merged

In [ ]:
# jan_station 列が "海老名" の行を表示
filtered_df = df_merged[df_merged['jan_station'] == '羽田空港第１・第２ターミナル']
filtered_df

In [ ]:
# # GeoJSONファイルのパスを指定して読み込む
# geojson_file = 'railroad_national_data_2023/N05-23_RailroadSection2.geojson'
# railroad_national_2023 = gpd.read_file(geojson_file)
# railroad_national_2023

In [ ]:
# tokyo_df = railroad_national_2023[railroad_national_2023['N05_003'].str.contains('東日本旅客鉄道', na=False)]
# tokyo_df

（４）駅データ.jpと路線名、路線数を結合

In [ ]:
# CSVファイルを読み込む
df_eki_data = pd.read_csv('eki_data_jp.csv')
df_eki_data

In [ ]:
# jan_station列の全角英数字を半角に変換（英字と数字のみ）
df_eki_data['jan_station'] = df_eki_data['jan_station'].apply(lambda x: jaconv.z2h(x, kana=False, ascii=True, digit=True))

In [ ]:
filtered_df = df_eki_data[df_eki_data['jan_station'] == '市ヶ谷']
filtered_df

In [ ]:
# 変更したい複数のstation_cdとjan_stationのペアを辞書として定義
replace_map = {
    (9980805, '松山駅前'): 'JR松山駅前',
    (2100139, 'みなみ寄居'): 'みなみ寄居<ホンダ寄居前>',
    (9950302, 'ジヤトコ前(ジヤトコ1地区前)'): 'ジヤトコ前',
    (9941319, 'トヨタモビリティ富山 Gスクエア五福前'): 'トヨタモビリティ富山Gスクエア五福前（五福末広町）',
    (1150217, '三ケ根'): '三ヶ根',
    (1171705, '下祇園'): '下祗園',
    (9942206, '中町(西町北)'): '中町（西町北）',
    (2800511, '二重橋前'): '二重橋前〈丸の内〉',
    (1151207, '五十鈴ケ丘'): '五十鈴ヶ丘',
    (1190504, '吉野ケ里公園'): '吉野ヶ里公園',
    (9991211, '吾妻'): '吾妻（雲仙市役所前）',
    (1172204, '四郎ケ原'): '四郎ヶ原',
    (9980609, '大手町'): '大手町駅前',
    (1150908, '島ケ原'): '島ヶ原',
    (9992219, '市立体育館前'): '市立体育館前（県庁通）',
    (2900111, '希望ヶ丘'): '希望ケ丘',
    (9971027, '広島港(宇品)'): '広島港（宇品）',
    (9971120, '広電西広島(己斐)'): '広電西広島（己斐）',
    (9971134, '廿日市市役所前(平良)'): '廿日市市役所前（平良）',
    (1170507, '弓ケ浜'): '弓ヶ浜',
    (1161716, '忍ケ丘'): '忍ヶ丘',
    (1132705, '成田空港(第1旅客ターミナル)'): '成田空港',
    (1150909, '月ケ瀬口'): '月ヶ瀬口',
    (9991202, '本諫早'): '本諫早（諫早市役所前）',
    (9970610, '東山・おかでんミュージアム駅'): '東山・おかでんミュージアム',
    (9980801, '松山市駅前'): '松山市駅',
    (1190627, '柳ケ浦'): '柳ヶ浦',
    (1170232, '梅ケ峠'): '梅ヶ峠',
    (1151020, '梅ケ谷'): '梅ヶ谷',
    (1161215, '櫛ケ浜'): '櫛ヶ浜',
    (2100217, '獨協大学前〈草加松原〉'): '獨協大学前<草加松原>',
    (9991215, '神代'): '神代（鍋島邸前）',
    (9992615, '神田(交通局前)'): '神田（交通局前）',
    (2400104, '笹塚'): '笹塚',
    (9941505, '粟島(大阪屋ショップ前)'): '粟島（大阪屋ショップ前）',
    (9980222, '綾川(イオンモール綾川)'): '綾川',
    (9941509, '蓮町(馬場記念公園前)'): '蓮町（馬場記念公園前）',
    (1180810, '西ケ方'): '西ヶ方',
    (9970606, '西大寺町・岡山芸術創造劇場ハレノワ前'): '西大寺町',
    (9941419, '第一イン新湊 クロスベイ前'): '西新湊',
    (3600101, '西鉄福岡(天神)'): '西鉄福岡（天神）',
    (9964808, '計算科学センター(神戸どうぶつ王国・「富岳」前)'): '計算科学センター',
    (9970303, '遙堪'): '遥堪',
    (1150311, '醒ケ井'): '醒ヶ井',
    (1150308, '関ケ原'): '関ヶ原',
    (1163209, '青野ケ原'): '青野ヶ原',
    (1141428, '駒ケ根'): '駒ヶ根',
    (9961504, '鶯の森'): '鴬の森',
    (2900109, '鶴ヶ峰'): '鶴ケ峰',
    (1162604, '鶴ケ丘'): '鶴ヶ丘',
    (1132907, '鹿島サッカースタジアム(臨)'): '鹿島サッカースタジアム',
    (2800615, '麹町'): '麴町',
    (9941515, '龍谷富山高校前(永楽町)'): '龍谷富山高校前（永楽町）',
    (2700207, '羽田空港第3ターミナル'): '羽田空港第３ターミナル',
    (9961817, 'あびこ'): '我孫子',
    (1132704, '空港第2ビル(第2旅客ターミナル)'): '空港第２ビル'
}

# 複数条件に基づいてjan_stationを更新
df_eki_data['jan_station'] = df_eki_data.apply(
    lambda row: replace_map.get((row['station_cd'], row['jan_station']), row['jan_station']),
    axis=1
)

In [ ]:
# 浜松町とモノレール浜松町を統合
# 'passengers_2000'から'passengers_2023'の列をリストにします
passenger_cols = [f'passengers_{year}' for year in range(2000, 2024)]

# 浜松町とモノレール浜松町のデータを取得します
hamamatsucho_row = df_merged[df_merged['jan_station'] == '浜松町']
monorail_hamamatsucho_row = df_merged[df_merged['jan_station'] == 'モノレール浜松町']

# 'passengers_2000'から'passengers_2023'の列について計算
for col in passenger_cols:
    hamamatsucho_val = hamamatsucho_row[col].values[0]
    monorail_val = monorail_hamamatsucho_row[col].values[0]

    if pd.isna(hamamatsucho_val) or pd.isna(monorail_val):
        # 片方がNaNの場合はNaNにする
        df_merged.loc[df_merged['jan_station'] == '浜松町', col] = np.nan
    else:
        # どちらもNaNでない場合は足し合わせる
        df_merged.loc[df_merged['jan_station'] == '浜松町', col] = hamamatsucho_val + monorail_val

# モノレール浜松町の行を削除
df_merged = df_merged[df_merged['jan_station'] != 'モノレール浜松町']

In [ ]:
# 都道府県名を追記
def calculate_distance(lat1, lon1, lat2, lon2):
    """2点間の距離をkm単位で計算する"""
    return geodesic((lat1, lon1), (lat2, lon2)).km

# 距離計算と条件に基づく結合を行う関数
def merge_with_distance(df1, df2, max_distance=10):
    merged = []
    for _, row1 in df1.iterrows():
        matches = df2[df2['jan_station'] == row1['jan_station']]
        if not matches.empty:
            for _, row2 in matches.iterrows():
                distance = calculate_distance(row1['latitude'], row1['longitude'],
                                              row2['lat'], row2['lon'])
                if distance <= max_distance:
                    merged.append({
                        **row1.to_dict(),
                        'connect_line_name': row2['connect_line_name'],
                        'connect_line_number': row2['connect_line_number'],
                        'connect_company_name': row2['connect_company_name'],
                        'connect_company_number': row2['connect_company_number'],
                        'pref_name': row2['pref_name']
                    })
                    break
            else:
                merged.append({
                    **row1.to_dict(),
                    'connect_line_name': np.nan,
                    'connect_line_number': np.nan,
                    'connect_company_name': np.nan,
                    'connect_company_number': np.nan,
                    'pref_name': np.nan
                })
        else:
            merged.append({
                **row1.to_dict(),
                'connect_line_name': np.nan,
                'connect_line_number': np.nan,
                'connect_company_name': np.nan,
                'connect_company_number': np.nan,
                'pref_name': np.nan
            })

    return pd.DataFrame(merged)

# データフレームの結合
df_merged = merge_with_distance(df_merged, df_eki_data)

# 結果の表示
df_merged

In [ ]:
# import requests
# import time

# # Google Maps APIキーを指定
# API_KEY = ""

# # 緯度経度から都道府県名を取得する関数
# def get_prefecture(lat, lon, api_key):
#     try:
#         time.sleep(0.2)  # 100ミリ秒のウェイト
#         # Google Maps Geocoding APIのエンドポイント
#         url = f"https://maps.googleapis.com/maps/api/geocode/json"

#         # パラメータの設定
#         params = {
#             "latlng": f"{lat},{lon}",
#             "key": api_key,
#             "language": "ja",
#             "result_type": "administrative_area_level_1"# 日本語での結果を指定
#         }

#         # APIリクエストを送信
#         response = requests.get(url, params=params)
#         response_data = response.json()

#         # 結果を解析
#         if response_data["status"] == "OK":
#             for component in response_data["results"][0]["address_components"]:
#                 if "administrative_area_level_1" in component["types"]:
#                     return component["long_name"]  # 都道府県名を返す

#         return None  # 都道府県が見つからなかった場合
#     except Exception as e:
#         print(f"Error: {e}")
#         return None

# # 新しい列に都道府県名を追加
# df_merged['prefecture'] = df_merged.apply(lambda row: get_prefecture(row['latitude'], row['longitude'], API_KEY), axis=1)

# # 結果の表示
# df_merged

In [ ]:
# jan_stationが'押上'の行に対して、connect_line_nameとconnect_line_numberを更新
df_merged.loc[df_merged['jan_station'] == '押上', ['connect_line_name', 'connect_line_number','connect_company_name','connect_company_number']] = ['東武伊勢崎線,東京メトロ半蔵門線,都営浅草線,京成押上線', 4,'東武鉄道,東京メトロ,東京都交通局,京成電鉄',4]
df_merged

In [ ]:
# 羽田空港の路線、路線数を修正
df_merged.loc[df_merged['jan_station'] == '羽田空港第１・第２ターミナル', ['connect_line_name', 'connect_line_number','connect_company_name','connect_company_number']] = ['東京モノレール,京急空港線', 2,'東京モノレール,京急電鉄',2]
df_merged

In [ ]:
# jan_stationが'市ケ谷'の行に対して、connect_line_nameとconnect_line_numberを更新
df_merged.loc[df_merged['jan_station'] == '市ケ谷', ['connect_line_name', 'connect_line_number','connect_company_name','connect_company_number']] = ['JR中央・総武線,東京メトロ有楽町線,東京メトロ南北線,都営新宿線', 4,'JR東日本,東京メトロ,東京都交通局',3]
df_merged

In [ ]:
# 乗降客数が2021、かつ、2022が0の行を削除
# df_merged = df_merged[~((df_merged['passengers_2021'] == 0) & (df_merged['passengers_2022'] == 0))]

In [ ]:
# NaNが含まれる行を抽出
nan_rows = df_merged[df_merged['connect_line_name'].isna()].reset_index(drop=True)
nan_rows

In [ ]:
# 条件に基づいて値を設定
df_merged.loc[df_merged['connect_line_name'].isna(), 'connect_line_name'] = df_merged['railway']
df_merged.loc[df_merged['connect_line_number'].isna(), 'connect_line_number'] = 1

In [ ]:
# 条件に基づいて値を設定
df_merged.loc[df_merged['connect_company_name'].isna(), 'connect_company_name'] = df_merged['operator']
df_merged.loc[df_merged['connect_company_number'].isna(), 'connect_company_number'] = 1

In [ ]:
# NaNが含まれる行を抽出
nan_rows = df_merged[df_merged['connect_company_name'].isna()].reset_index(drop=True)
nan_rows

In [ ]:
df_merged.reset_index(drop=True)

In [ ]:
# CSVファイルを読み込む
df_prefecture = pd.read_csv('prefecture_utf-8.csv')
df_prefecture

In [ ]:
# インデックス3346の市ヶ谷を削除（応急処置）
df_prefecture = df_prefecture.drop(index=3346).reset_index(drop=True)
df_prefecture

In [ ]:
# 都道府県データを横結合
df_merged = pd.concat([df_merged, df_prefecture[['prefecture']]], axis=1).reset_index(drop=True)
df_merged

In [ ]:
# prefecture列にNaNが含まれる行の数
nan_count = df_merged['pref_name'].isna().sum()

print(f"pref_name'列のNaNの数: {nan_count}")

In [ ]:
df_merged['pref_name'] = df_merged['pref_name'].fillna(df_merged['prefecture'])

# 処理後にNaNが残っているか確認
nan_count = df_merged['pref_name'].isna().sum()
print(f"処理後のpref_name列のNaNの数: {nan_count}")

In [ ]:
# latitude列のNaNを含む行を削除し、インデックスを振り直す
df_merged = df_merged.dropna(subset=['latitude']).reset_index(drop=True)

（５）人口データと境界データの統合

In [ ]:
def load_population_data(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*.txt"))
    df_list = []
    for file in all_files:
        df = pd.read_csv(
            file,
            header=0,              # 最初のヘッダー行のみを使用
            skiprows=[1],          # 2行目のヘッダーをスキップ
            encoding='cp932',      # 日本語のエンコーディング
            na_values=['*', '']    # 欠損値の扱い
        )
        df_list.append(df)
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

# フォルダのパスを指定
folder_path = 'population_1km_2020'

# データフレームを取得
df_population_2020 = load_population_data(folder_path)

df_population_2020

In [ ]:
#境界データ
# boundary_data_1km フォルダのパス
folder_path = 'boundary_data_1km'

# フォルダ内のすべての.shpファイルを取得
shapefiles = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.shp')]

# すべてのシェープファイルを読み込み、リストに格納
gdf_list = [gpd.read_file(shp) for shp in shapefiles]

# すべてのGeoDataFrameを一つのGeoDataFrameに結合
df_boundary = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

# 結果のGeoDataFrameを表示
df_boundary

In [ ]:
df_boundary['KEY_CODE'] = df_boundary['KEY_CODE'].astype(int)

In [ ]:
# データフレームを横結合する
df_population_boundary_2020 = df_population_2020.merge(df_boundary[['KEY_CODE', 'geometry']],
                               on='KEY_CODE',
                               how='left')
df_population_boundary_2020

In [ ]:
# 日本に適したカスタム等積投影法を定義（修正済み）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# df_population_boundary_2020をGeoDataFrameに変換し、CRSをJGD2011に設定
df_population_boundary_2020 = gpd.GeoDataFrame(df_population_boundary_2020, geometry='geometry', crs='EPSG:6668')

# df_mergedの緯度・経度からPoint型のgeometryを作成し、CRSをJGD2011に設定
df_merged['geometry'] = gpd.points_from_xy(df_merged['longitude'], df_merged['latitude'])
df_merged = gpd.GeoDataFrame(df_merged, geometry='geometry', crs='EPSG:6668')

# df_mergedに'station_id'列を追加（インデックスを使用）
df_merged = df_merged.reset_index().rename(columns={'index': 'station_id'})

# 面積計算とバッファ作成のために、両方のデータフレームをカスタム等積投影法に変換
df_population_boundary_2020_equal_area = df_population_boundary_2020.to_crs(custom_equal_area_crs)
df_merged_equal_area = df_merged.to_crs(custom_equal_area_crs)

# 各駅の周囲2kmのバッファを作成
df_merged_equal_area['buffer_2km'] = df_merged_equal_area.buffer(2000)

# 'buffer_2km'をアクティブなジオメトリ列として設定
buffers = df_merged_equal_area[['station_id', 'buffer_2km']].copy()
buffers = buffers.set_geometry('buffer_2km')

# 人口ポリゴンの面積を計算
df_population_boundary_2020_equal_area['original_area'] = df_population_boundary_2020_equal_area.area

# バッファと人口ポリゴンの交差部分を計算
intersection = gpd.overlay(
    df_population_boundary_2020_equal_area,
    buffers,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection['intersection_area'] = intersection.area

# 面積比を計算
intersection['area_ratio'] = intersection['intersection_area'] / intersection['original_area']

# 面積比を用いて人口を按分
intersection['population_within_2km_2020'] = intersection['T001140001'] * intersection['area_ratio']

# 各駅ごとに按分した人口を合計
population_by_station = intersection.groupby('station_id')['population_within_2km_2020'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと人口データをマージし、NaNを0で埋める
population_by_station = all_station_ids.merge(population_by_station, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 各駅の周囲1kmのバッファを作成
df_merged_equal_area['buffer_1km'] = df_merged_equal_area.buffer(1000)

# 'buffer_1km'をアクティブなジオメトリ列として設定
buffers_1km = df_merged_equal_area[['station_id', 'buffer_1km']].copy()
buffers_1km = buffers_1km.set_geometry('buffer_1km')

# バッファ（1km）と人口ポリゴンの交差部分を計算
intersection_1km = gpd.overlay(
    df_population_boundary_2020_equal_area,
    buffers_1km,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_1km['intersection_area_1km'] = intersection_1km.area

# 面積比を計算
intersection_1km['area_ratio_1km'] = intersection_1km['intersection_area_1km'] / intersection_1km['original_area']

# 面積比を用いて1km圏内人口を按分
intersection_1km['population_within_1km_2020'] = intersection_1km['T001140001'] * intersection_1km['area_ratio_1km']

# 各駅ごとに按分した人口を合計
population_by_station_1km = intersection_1km.groupby('station_id')['population_within_1km_2020'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_1km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと1km人口データをマージし、NaNを0で埋める
population_by_station_1km = all_station_ids_1km.merge(population_by_station_1km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_1km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 各駅の周囲5kmのバッファを作成
df_merged_equal_area['buffer_5km'] = df_merged_equal_area.buffer(5000)

# 'buffer_5km'をアクティブなジオメトリ列として設定
buffers_5km = df_merged_equal_area[['station_id', 'buffer_5km']].copy()
buffers_5km = buffers_5km.set_geometry('buffer_5km')

# バッファ（5km）と人口ポリゴンの交差部分を計算
intersection_5km = gpd.overlay(
    df_population_boundary_2020_equal_area,
    buffers_5km,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_5km['intersection_area_5km'] = intersection_5km.area

# 面積比を計算
intersection_5km['area_ratio_5km'] = intersection_5km['intersection_area_5km'] / intersection_5km['original_area']

# 面積比を用いて5km圏内人口を按分
intersection_5km['population_within_5km_2020'] = intersection_5km['T001140001'] * intersection_5km['area_ratio_5km']

# 各駅ごとに按分した人口を合計
population_by_station_5km = intersection_5km.groupby('station_id')['population_within_5km_2020'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_5km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと5km人口データをマージし、NaNを0で埋める
population_by_station_5km = all_station_ids_5km.merge(population_by_station_5km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_5km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 各駅の周囲10kmのバッファを作成
df_merged_equal_area['buffer_10km'] = df_merged_equal_area.buffer(10000)

# 'buffer_10km'をアクティブなジオメトリ列として設定
buffers_10km = df_merged_equal_area[['station_id', 'buffer_10km']].copy()
buffers_10km = buffers_10km.set_geometry('buffer_10km')

# バッファ（10km）と人口ポリゴンの交差部分を計算
intersection_10km = gpd.overlay(
    df_population_boundary_2020_equal_area,
    buffers_10km,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_10km['intersection_area_10km'] = intersection_10km.area

# 面積比を計算
intersection_10km['area_ratio_10km'] = intersection_10km['intersection_area_10km'] / intersection_10km['original_area']

# 面積比を用いて10km圏内人口を按分
intersection_10km['population_within_10km_2020'] = intersection_10km['T001140001'] * intersection_10km['area_ratio_10km']

# 各駅ごとに按分した人口を合計
population_by_station_10km = intersection_10km.groupby('station_id')['population_within_10km_2020'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_10km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと10km人口データをマージし、NaNを0で埋める
population_by_station_10km = all_station_ids_10km.merge(population_by_station_10km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_10km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
nan_counts = df_merged.isna().sum()

print(nan_counts)

In [ ]:
def load_population_data(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*.txt"))
    df_list = []
    for file in all_files:
        df = pd.read_csv(
            file,
            header=0,              # 最初のヘッダー行のみを使用
            skiprows=[1],          # 2行目のヘッダーをスキップ
            encoding='cp932',      # 日本語のエンコーディング
            na_values=['*', '']    # 欠損値の扱い
        )
        df_list.append(df)
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

# フォルダのパスを指定
folder_path = 'population_1km_2015'

# データフレームを取得
df_population_2015 = load_population_data(folder_path)

df_population_2015

In [ ]:
# データフレームを横結合する
df_population_boundary_2015 = df_population_2015.merge(df_boundary[['KEY_CODE', 'geometry']],
                               on='KEY_CODE',
                               how='left')
df_population_boundary_2015

In [ ]:
# 日本に適したカスタム等積投影法を定義（修正済み）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# df_population_boundary_2015をGeoDataFrameに変換し、CRSをJGD2011に設定
df_population_boundary_2015 = gpd.GeoDataFrame(df_population_boundary_2015, geometry='geometry', crs='EPSG:6668')

# 面積計算とバッファ作成のために、両方のデータフレームをカスタム等積投影法に変換
df_population_boundary_2015_equal_area = df_population_boundary_2015.to_crs(custom_equal_area_crs)

# 人口ポリゴンの面積を計算
df_population_boundary_2015_equal_area['original_area'] = df_population_boundary_2015_equal_area.area

# バッファと人口ポリゴンの交差部分を計算
intersection = gpd.overlay(
    df_population_boundary_2015_equal_area,
    buffers,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection['intersection_area'] = intersection.area

# 面積比を計算
intersection['area_ratio'] = intersection['intersection_area'] / intersection['original_area']

# 面積比を用いて人口を按分
intersection['population_within_2km_2015'] = intersection['T000846001'] * intersection['area_ratio']

# 各駅ごとに按分した人口を合計
population_by_station = intersection.groupby('station_id')['population_within_2km_2015'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと人口データをマージし、NaNを0で埋める
population_by_station = all_station_ids.merge(population_by_station, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 1kmバッファと2015年人口ポリゴンの交差部分を計算
intersection_1km_2015 = gpd.overlay(
    df_population_boundary_2015_equal_area,
    buffers_1km,
    how='intersection'
)

# station_id列の確認とリネーム
if 'station_id_1' in intersection_1km_2015.columns:
    intersection_1km_2015 = intersection_1km_2015.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_1km_2015.columns:
    intersection_1km_2015 = intersection_1km_2015.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_1km_2015['intersection_area_1km'] = intersection_1km_2015.area

# 面積比を計算
intersection_1km_2015['area_ratio_1km'] = intersection_1km_2015['intersection_area_1km'] / intersection_1km_2015['original_area']

# 面積比を用いて1km圏内人口を按分
intersection_1km_2015['population_within_1km_2015'] = intersection_1km_2015['T000846001'] * intersection_1km_2015['area_ratio_1km']

# 各駅ごとに按分した人口を合計
population_by_station_1km_2015 = intersection_1km_2015.groupby('station_id')['population_within_1km_2015'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_1km_2015 = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと1km人口データをマージし、NaNを0で埋める
population_by_station_1km_2015 = all_station_ids_1km_2015.merge(population_by_station_1km_2015, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_1km_2015, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 5kmバッファと2015年人口ポリゴンの交差部分を計算
intersection_5km_2015 = gpd.overlay(
    df_population_boundary_2015_equal_area,
    buffers_5km,
    how='intersection'
)

# station_id列の確認とリネーム
if 'station_id_1' in intersection_5km_2015.columns:
    intersection_5km_2015 = intersection_5km_2015.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_5km_2015.columns:
    intersection_5km_2015 = intersection_5km_2015.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_5km_2015['intersection_area_5km'] = intersection_5km_2015.area

# 面積比を計算
intersection_5km_2015['area_ratio_5km'] = intersection_5km_2015['intersection_area_5km'] / intersection_5km_2015['original_area']

# 面積比を用いて1km圏内人口を按分
intersection_5km_2015['population_within_5km_2015'] = intersection_5km_2015['T000846001'] * intersection_5km_2015['area_ratio_5km']

# 各駅ごとに按分した人口を合計
population_by_station_5km_2015 = intersection_5km_2015.groupby('station_id')['population_within_5km_2015'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_5km_2015 = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと5km人口データをマージし、NaNを0で埋める
population_by_station_5km_2015 = all_station_ids_5km_2015.merge(population_by_station_5km_2015, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_5km_2015, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 10kmバッファと2015年人口ポリゴンの交差部分を計算
intersection_10km_2015 = gpd.overlay(
    df_population_boundary_2015_equal_area,
    buffers_10km,
    how='intersection'
)

# station_id列の確認とリネーム
if 'station_id_1' in intersection_10km_2015.columns:
    intersection_10km_2015 = intersection_10km_2015.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_10km_2015.columns:
    intersection_10km_2015 = intersection_10km_2015.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_10km_2015['intersection_area_10km'] = intersection_10km_2015.area

# 面積比を計算
intersection_10km_2015['area_ratio_10km'] = intersection_10km_2015['intersection_area_10km'] / intersection_10km_2015['original_area']

# 面積比を用いて1km圏内人口を按分
intersection_10km_2015['population_within_10km_2015'] = intersection_10km_2015['T000846001'] * intersection_10km_2015['area_ratio_10km']

# 各駅ごとに按分した人口を合計
population_by_station_10km_2015 = intersection_10km_2015.groupby('station_id')['population_within_10km_2015'].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_10km_2015 = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと10km人口データをマージし、NaNを0で埋める
population_by_station_10km_2015 = all_station_ids_10km_2015.merge(population_by_station_10km_2015, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_10km_2015, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
nan_counts = df_merged.isna().sum()

print(nan_counts)

In [ ]:
# latitude列がNaNの行を抽出
nan_latitude_rows = df_merged[df_merged['latitude'].isna()]

# 抽出した結果を確認
nan_latitude_rows

（５）地価

In [ ]:
# 日本に適したカスタム等積投影法を定義（人口計算コードと同じ）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# ファイル名、地価の列、結合する列名をリストとして定義します
file_names = [
    'L01-24.geojson', 'L01-23.geojson', 'L01-22.geojson', 'L01-21.geojson', 'L01-20.geojson',
    'L01-19.geojson', 'L01-18.geojson', 'L01-17.shp', 'L01-16.shp', 'L01-15.shp'
]

price_columns = [
    'L01_008', 'L01_006', 'L01_006', 'L01_006', 'L01_006',
    'L01_006', 'L01_006', 'L01_006', 'L01_006', 'L01_006'
]

new_column_names = [
    'average_land_price_within_2km_2024', 'average_land_price_within_2km_2023', 'average_land_price_within_2km_2022',
    'average_land_price_within_2km_2021', 'average_land_price_within_2km_2020', 'average_land_price_within_2km_2019',
    'average_land_price_within_2km_2018', 'average_land_price_within_2km_2017', 'average_land_price_within_2km_2016',
    'average_land_price_within_2km_2015'
]

# 各ファイルについてループ処理を行います
for file_name, price_col, new_col in zip(file_names, price_columns, new_column_names):
    file_path = f'land_of_price/{file_name}'
    file_extension = file_name.split('.')[-1]

    if file_extension == 'geojson' or file_extension == 'shp':
        # 地価データを読み込みます
        df_price_of_land = gpd.read_file(file_path)

        # price_colが存在しない場合、.dbfファイルを読み込んで結合します
        if price_col not in df_price_of_land.columns and file_extension == 'shp':
            dbf_path = file_path.replace('.shp', '.dbf')
            from dbfread import DBF
            dbf = DBF(dbf_path, encoding='cp932')
            df_dbf = pd.DataFrame(iter(dbf))
            # インデックスで結合
            df_price_of_land = df_price_of_land.merge(df_dbf, left_index=True, right_index=True)
    else:
        print(f"Unsupported file type: {file_extension}")
        continue

    # 座標系を設定
    if df_price_of_land.crs is None:
        df_price_of_land.crs = 'EPSG:6668'

    # 距離計算と面積計算のため、カスタム等積投影法に変換
    df_price_of_land = df_price_of_land.to_crs(custom_equal_area_crs)

    # 地価データの列を数値型に変換（不要な文字を除去）
    df_price_of_land[price_col] = df_price_of_land[price_col].astype(str)
    df_price_of_land[price_col] = df_price_of_land[price_col].str.replace(',', '')
    df_price_of_land[price_col] = pd.to_numeric(df_price_of_land[price_col], errors='coerce')

    # 駅のバッファと地価データの空間結合を行う
    df_land_price_within_buffers = gpd.sjoin(df_price_of_land, buffers, how='inner', predicate='within')

    # 駅ごとに地価の平均を計算
    average_land_price_by_station = df_land_price_within_buffers.groupby('station_id')[price_col].mean().reset_index()

    # バッファ内に地価データがない駅を特定
    stations_with_land_price = average_land_price_by_station['station_id'].unique()
    stations_without_land_price = df_merged[~df_merged['station_id'].isin(stations_with_land_price)]

    # バッファ内に地価データがない駅について、最寄りの地価ポイントを探す
    if not stations_without_land_price.empty:
        from scipy.spatial import cKDTree

        # 駅の座標を取得
        stations_without_land_price = stations_without_land_price.to_crs(custom_equal_area_crs)
        station_coords = np.array(list(zip(stations_without_land_price.geometry.x, stations_without_land_price.geometry.y)))

        # 地価ポイントの座標を取得
        land_price_coords = np.array(list(zip(df_price_of_land.geometry.x, df_price_of_land.geometry.y)))

        # KDTreeを作成
        tree = cKDTree(land_price_coords)

        # 最近傍の地価ポイントを検索
        distances, indices = tree.query(station_coords, k=1)

        # 最近傍の地価を取得
        nearest_land_prices = df_price_of_land.iloc[indices][price_col].values

        # 結果をデータフレームにまとめる
        nearest_land_price_df = pd.DataFrame({
            'station_id': stations_without_land_price['station_id'],
            new_col: nearest_land_prices
        })

        # 平均地価と最近傍地価を結合
        land_price_by_station = pd.concat([
            average_land_price_by_station.rename(columns={price_col: new_col}),
            nearest_land_price_df
        ], ignore_index=True)
    else:
        # すべての駅がバッファ内に地価データを持つ場合
        land_price_by_station = average_land_price_by_station.rename(columns={price_col: new_col})

    # 結果をdf_mergedにマージ
    df_merged = df_merged.merge(land_price_by_station[['station_id', new_col]], on='station_id', how='left')

# 結果を確認
df_merged

In [ ]:
# 1kmバッファを利用して地価を計算する列リストを定義
file_names_1km = [
    'L01-24.geojson', 'L01-23.geojson', 'L01-22.geojson', 'L01-21.geojson', 'L01-20.geojson',
    'L01-19.geojson', 'L01-18.geojson', 'L01-17.shp', 'L01-16.shp', 'L01-15.shp'
]

price_columns_1km = [
    'L01_008', 'L01_006', 'L01_006', 'L01_006', 'L01_006',
    'L01_006', 'L01_006', 'L01_006', 'L01_006', 'L01_006'
]

new_column_names_1km = [
    'average_land_price_within_1km_2024', 'average_land_price_within_1km_2023', 'average_land_price_within_1km_2022',
    'average_land_price_within_1km_2021', 'average_land_price_within_1km_2020', 'average_land_price_within_1km_2019',
    'average_land_price_within_1km_2018', 'average_land_price_within_1km_2017', 'average_land_price_within_1km_2016',
    'average_land_price_within_1km_2015'
]

for file_name, price_col, new_col in zip(file_names_1km, price_columns_1km, new_column_names_1km):
    file_path = f'land_of_price/{file_name}'
    file_extension = file_name.split('.')[-1]

    if file_extension == 'geojson' or file_extension == 'shp':
        # 地価データを読み込み
        df_price_of_land = gpd.read_file(file_path)

        # price_colが存在しない場合、.dbfファイルを読み込み結合
        if price_col not in df_price_of_land.columns and file_extension == 'shp':
            dbf_path = file_path.replace('.shp', '.dbf')
            from dbfread import DBF
            dbf = DBF(dbf_path, encoding='cp932')
            df_dbf = pd.DataFrame(iter(dbf))
            df_price_of_land = df_price_of_land.merge(df_dbf, left_index=True, right_index=True)
    else:
        print(f"Unsupported file type: {file_extension}")
        continue

    # CRSが未設定の場合はEPSG:6668を設定
    if df_price_of_land.crs is None:
        df_price_of_land.crs = 'EPSG:6668'

    # カスタム等積投影法に変換
    df_price_of_land = df_price_of_land.to_crs(custom_equal_area_crs)

    # 地価列を数値型に変換
    df_price_of_land[price_col] = df_price_of_land[price_col].astype(str)
    df_price_of_land[price_col] = df_price_of_land[price_col].str.replace(',', '')
    df_price_of_land[price_col] = pd.to_numeric(df_price_of_land[price_col], errors='coerce')

    # 1kmバッファと地価データの空間結合
    df_land_price_within_buffers_1km = gpd.sjoin(df_price_of_land, buffers_1km, how='inner', predicate='within')

    # 駅ごとに地価の平均を計算
    average_land_price_by_station_1km = df_land_price_within_buffers_1km.groupby('station_id')[price_col].mean().reset_index()

    # バッファ内に地価データがない駅を特定
    stations_with_land_price_1km = average_land_price_by_station_1km['station_id'].unique()
    stations_without_land_price_1km = df_merged[~df_merged['station_id'].isin(stations_with_land_price_1km)]

    # バッファ内に地価データがない場合は、最寄りの地価ポイントを探す
    if not stations_without_land_price_1km.empty:
        from scipy.spatial import cKDTree

        # 駅の座標を取得（カスタム等積投影法）
        stations_without_land_price_1km = stations_without_land_price_1km.to_crs(custom_equal_area_crs)
        station_coords_1km = np.array(list(zip(stations_without_land_price_1km.geometry.x, stations_without_land_price_1km.geometry.y)))

        # 地価ポイントの座標を取得
        land_price_coords_1km = np.array(list(zip(df_price_of_land.geometry.x, df_price_of_land.geometry.y)))

        # KDTreeを作成
        tree_1km = cKDTree(land_price_coords_1km)

        # 最近傍の地価ポイントを検索
        distances_1km, indices_1km = tree_1km.query(station_coords_1km, k=1)

        # 最近傍の地価を取得
        nearest_land_prices_1km = df_price_of_land.iloc[indices_1km][price_col].values

        # 結果をデータフレームにまとめ
        nearest_land_price_df_1km = pd.DataFrame({
            'station_id': stations_without_land_price_1km['station_id'],
            new_col: nearest_land_prices_1km
        })

        # 平均地価と最近傍地価を結合
        land_price_by_station_1km = pd.concat([
            average_land_price_by_station_1km.rename(columns={price_col: new_col}),
            nearest_land_price_df_1km
        ], ignore_index=True)
    else:
        # すべての駅がバッファ内に地価データを持つ場合
        land_price_by_station_1km = average_land_price_by_station_1km.rename(columns={price_col: new_col})

    # 結果をdf_mergedにマージ
    df_merged = df_merged.merge(land_price_by_station_1km[['station_id', new_col]], on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 5kmバッファを利用して地価を計算する列リストを定義
file_names_5km = [
    'L01-24.geojson', 'L01-23.geojson', 'L01-22.geojson', 'L01-21.geojson', 'L01-20.geojson',
    'L01-19.geojson', 'L01-18.geojson', 'L01-17.shp', 'L01-16.shp', 'L01-15.shp'
]

price_columns_5km = [
    'L01_008', 'L01_006', 'L01_006', 'L01_006', 'L01_006',
    'L01_006', 'L01_006', 'L01_006', 'L01_006', 'L01_006'
]

new_column_names_5km = [
    'average_land_price_within_5km_2024', 'average_land_price_within_5km_2023', 'average_land_price_within_5km_2022',
    'average_land_price_within_5km_2021', 'average_land_price_within_5km_2020', 'average_land_price_within_5km_2019',
    'average_land_price_within_5km_2018', 'average_land_price_within_5km_2017', 'average_land_price_within_5km_2016',
    'average_land_price_within_5km_2015'
]

for file_name, price_col, new_col in zip(file_names_5km, price_columns_5km, new_column_names_5km):
    file_path = f'land_of_price/{file_name}'
    file_extension = file_name.split('.')[-1]

    if file_extension == 'geojson' or file_extension == 'shp':
        # 地価データを読み込み
        df_price_of_land = gpd.read_file(file_path)

        # price_colが存在しない場合、.dbfファイルを読み込み結合
        if price_col not in df_price_of_land.columns and file_extension == 'shp':
            dbf_path = file_path.replace('.shp', '.dbf')
            from dbfread import DBF
            dbf = DBF(dbf_path, encoding='cp932')
            df_dbf = pd.DataFrame(iter(dbf))
            df_price_of_land = df_price_of_land.merge(df_dbf, left_index=True, right_index=True)
    else:
        print(f"Unsupported file type: {file_extension}")
        continue

    # CRSが未設定の場合はEPSG:6668を設定
    if df_price_of_land.crs is None:
        df_price_of_land.crs = 'EPSG:6668'

    # カスタム等積投影法に変換
    df_price_of_land = df_price_of_land.to_crs(custom_equal_area_crs)

    # 地価列を数値型に変換
    df_price_of_land[price_col] = df_price_of_land[price_col].astype(str)
    df_price_of_land[price_col] = df_price_of_land[price_col].str.replace(',', '')
    df_price_of_land[price_col] = pd.to_numeric(df_price_of_land[price_col], errors='coerce')

    # 5kmバッファと地価データの空間結合
    df_land_price_within_buffers_5km = gpd.sjoin(df_price_of_land, buffers_5km, how='inner', predicate='within')

    # 駅ごとに地価の平均を計算
    average_land_price_by_station_5km = df_land_price_within_buffers_5km.groupby('station_id')[price_col].mean().reset_index()

    # バッファ内に地価データがない駅を特定
    stations_with_land_price_5km = average_land_price_by_station_5km['station_id'].unique()
    stations_without_land_price_5km = df_merged[~df_merged['station_id'].isin(stations_with_land_price_5km)]

    # バッファ内に地価データがない場合は、最寄りの地価ポイントを探す
    if not stations_without_land_price_5km.empty:
        from scipy.spatial import cKDTree

        # 駅の座標を取得（カスタム等積投影法）
        stations_without_land_price_5km = stations_without_land_price_5km.to_crs(custom_equal_area_crs)
        station_coords_5km = np.array(list(zip(stations_without_land_price_5km.geometry.x, stations_without_land_price_5km.geometry.y)))

        # 地価ポイントの座標を取得
        land_price_coords_5km = np.array(list(zip(df_price_of_land.geometry.x, df_price_of_land.geometry.y)))

        # KDTreeを作成
        tree_5km = cKDTree(land_price_coords_5km)

        # 最近傍の地価ポイントを検索
        distances_5km, indices_5km = tree_5km.query(station_coords_5km, k=1)

        # 最近傍の地価を取得
        nearest_land_prices_5km = df_price_of_land.iloc[indices_5km][price_col].values

        # 結果をデータフレームにまとめ
        nearest_land_price_df_5km = pd.DataFrame({
            'station_id': stations_without_land_price_5km['station_id'],
            new_col: nearest_land_prices_5km
        })

        # 平均地価と最近傍地価を結合
        land_price_by_station_5km = pd.concat([
            average_land_price_by_station_5km.rename(columns={price_col: new_col}),
            nearest_land_price_df_5km
        ], ignore_index=True)
    else:
        # すべての駅がバッファ内に地価データを持つ場合
        land_price_by_station_5km = average_land_price_by_station_5km.rename(columns={price_col: new_col})

    # 結果をdf_mergedにマージ
    df_merged = df_merged.merge(land_price_by_station_5km[['station_id', new_col]], on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 10kmバッファを利用して地価を計算する列リストを定義
file_names_10km = [
    'L01-24.geojson', 'L01-23.geojson', 'L01-22.geojson', 'L01-21.geojson', 'L01-20.geojson',
    'L01-19.geojson', 'L01-18.geojson', 'L01-17.shp', 'L01-16.shp', 'L01-15.shp'
]

price_columns_10km = [
    'L01_008', 'L01_006', 'L01_006', 'L01_006', 'L01_006',
    'L01_006', 'L01_006', 'L01_006', 'L01_006', 'L01_006'
]

new_column_names_10km = [
    'average_land_price_within_10km_2024', 'average_land_price_within_10km_2023', 'average_land_price_within_10km_2022',
    'average_land_price_within_10km_2021', 'average_land_price_within_10km_2020', 'average_land_price_within_10km_2019',
    'average_land_price_within_10km_2018', 'average_land_price_within_10km_2017', 'average_land_price_within_10km_2016',
    'average_land_price_within_10km_2015'
]

for file_name, price_col, new_col in zip(file_names_10km, price_columns_10km, new_column_names_10km):
    file_path = f'land_of_price/{file_name}'
    file_extension = file_name.split('.')[-1]

    if file_extension == 'geojson' or file_extension == 'shp':
        # 地価データを読み込み
        df_price_of_land = gpd.read_file(file_path)

        # price_colが存在しない場合、.dbfファイルを読み込み結合
        if price_col not in df_price_of_land.columns and file_extension == 'shp':
            dbf_path = file_path.replace('.shp', '.dbf')
            from dbfread import DBF
            dbf = DBF(dbf_path, encoding='cp932')
            df_dbf = pd.DataFrame(iter(dbf))
            df_price_of_land = df_price_of_land.merge(df_dbf, left_index=True, right_index=True)
    else:
        print(f"Unsupported file type: {file_extension}")
        continue

    # CRSが未設定の場合はEPSG:6668を設定
    if df_price_of_land.crs is None:
        df_price_of_land.crs = 'EPSG:6668'

    # カスタム等積投影法に変換
    df_price_of_land = df_price_of_land.to_crs(custom_equal_area_crs)

    # 地価列を数値型に変換
    df_price_of_land[price_col] = df_price_of_land[price_col].astype(str)
    df_price_of_land[price_col] = df_price_of_land[price_col].str.replace(',', '')
    df_price_of_land[price_col] = pd.to_numeric(df_price_of_land[price_col], errors='coerce')

    # 10kmバッファと地価データの空間結合
    df_land_price_within_buffers_10km = gpd.sjoin(df_price_of_land, buffers_10km, how='inner', predicate='within')

    # 駅ごとに地価の平均を計算
    average_land_price_by_station_10km = df_land_price_within_buffers_10km.groupby('station_id')[price_col].mean().reset_index()

    # バッファ内に地価データがない駅を特定
    stations_with_land_price_10km = average_land_price_by_station_10km['station_id'].unique()
    stations_without_land_price_10km = df_merged[~df_merged['station_id'].isin(stations_with_land_price_10km)]

    # バッファ内に地価データがない場合は、最寄りの地価ポイントを探す
    if not stations_without_land_price_10km.empty:
        from scipy.spatial import cKDTree

        # 駅の座標を取得（カスタム等積投影法）
        stations_without_land_price_10km = stations_without_land_price_10km.to_crs(custom_equal_area_crs)
        station_coords_10km = np.array(list(zip(stations_without_land_price_10km.geometry.x, stations_without_land_price_10km.geometry.y)))

        # 地価ポイントの座標を取得
        land_price_coords_10km = np.array(list(zip(df_price_of_land.geometry.x, df_price_of_land.geometry.y)))

        # KDTreeを作成
        tree_10km = cKDTree(land_price_coords_10km)

        # 最近傍の地価ポイントを検索
        distances_10km, indices_10km = tree_10km.query(station_coords_10km, k=1)

        # 最近傍の地価を取得
        nearest_land_prices_10km = df_price_of_land.iloc[indices_10km][price_col].values

        # 結果をデータフレームにまとめ
        nearest_land_price_df_10km = pd.DataFrame({
            'station_id': stations_without_land_price_10km['station_id'],
            new_col: nearest_land_prices_10km
        })

        # 平均地価と最近傍地価を結合
        land_price_by_station_10km = pd.concat([
            average_land_price_by_station_10km.rename(columns={price_col: new_col}),
            nearest_land_price_df_10km
        ], ignore_index=True)
    else:
        # すべての駅がバッファ内に地価データを持つ場合
        land_price_by_station_10km = average_land_price_by_station_10km.rename(columns={price_col: new_col})

    # 結果をdf_mergedにマージ
    df_merged = df_merged.merge(land_price_by_station_10km[['station_id', new_col]], on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
nan_counts = df_merged.isna().sum()

print(nan_counts)

（６）バス停

In [ ]:
import zipfile

# ZIPファイルが保存されているフォルダのパス
folder_path = 'bus_2022'

# 縦結合するための空のGeoDataFrameを作成
bus_2022_gdf = gpd.GeoDataFrame()

# フォルダ内のすべてのzipファイルを取得
for zip_filename in os.listdir(folder_path):
    if zip_filename.endswith('.zip'):
        zip_filepath = os.path.join(folder_path, zip_filename)

        # ZIPファイルを開く
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            # ZIPファイル内のすべてのファイルをリストアップ
            for file in zip_ref.namelist():
                # geojsonファイルを探す
                if file.endswith('.geojson'):
                    # 一時ファイルとして解凍して読み込む
                    with zip_ref.open(file) as geojson_file:
                        gdf = gpd.read_file(geojson_file)
                        # GeoDataFrameを縦結合
                        bus_2022_gdf = gpd.GeoDataFrame(pd.concat([bus_2022_gdf, gdf], ignore_index=True))

# 結果を確認
bus_2022_gdf

In [ ]:
# カラム名を指定して変更する
bus_2022_gdf = bus_2022_gdf.rename(columns={
    'P11_001': 'bus_stop_name',
    'P11_002': 'company_name',
    'P11_003_01': 'route_name'
})

In [ ]:
bus_2022_gdf

In [ ]:
# ベースディレクトリ
base_dir = 'highway_bus_2023'

# すべてのgeojsonファイルを格納するリスト
all_geojson_data = []

# フォルダ内のすべてのサブフォルダを探索
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # geojsonファイルのみを対象にする
        if file.endswith('.geojson'):
            file_path = os.path.join(root, file)
            # geojsonファイルを読み込む
            gdf = gpd.read_file(file_path)
            # データをリストに追加
            all_geojson_data.append(gdf)

# すべてのgeojsonデータを縦に結合
highway_bus_2023 = gpd.GeoDataFrame(pd.concat(all_geojson_data, ignore_index=True))

In [ ]:
# カラム名を指定して変更する
highway_bus_2023  = highway_bus_2023.rename(columns={
    'P36_001': 'bus_stop_name',
    'P36_002': 'company_name',
    'P36_003_01': 'route_name'
})

highway_bus_2023

In [ ]:
# bus_2022_gdfとhighway_bus_2023から必要なカラムのみ選択
bus_2022_selected = bus_2022_gdf[['bus_stop_name', 'company_name', 'route_name','geometry']]
highway_bus_2023_selected = highway_bus_2023[['bus_stop_name', 'company_name', 'route_name','geometry']]

# 縦に結合
bus_2022_2023_gdf = pd.concat([bus_2022_selected, highway_bus_2023_selected], ignore_index=True)

bus_2022_2023_gdf

In [ ]:
bus_2022_2023_gdf_unique = bus_2022_2023_gdf.drop_duplicates(subset=['bus_stop_name', 'geometry'], keep='first')
bus_2022_2023_gdf_unique

In [ ]:
# bus_2010フォルダのパス
folder_path = 'bus_2010'

# bus_2010フォルダのパス
folder_path = 'bus_2010'

# すべてのzipファイルのパスを取得
zip_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.zip')]

# SHPファイルを格納するリスト
shp_list = []

# zipファイルを展開してSHPファイルと関連ファイルを読み込む
for zip_file in zip_files:
    with zipfile.ZipFile(zip_file, 'r') as z:
        # zipファイル内のファイルリストを取得
        z.extractall('temp')  # すべてのファイルを一時フォルダに展開

        # 展開されたフォルダ内のSHPファイルを探す
        for root, dirs, files in os.walk('temp'):
            for file_name in files:
                if file_name.endswith('.shp'):
                    shp_path = os.path.join(root, file_name)
                    shp = gpd.read_file(shp_path)
                    shp_list.append(shp)

# すべてのSHPファイルを縦結合
bus_2010 = gpd.GeoDataFrame(pd.concat(shp_list, ignore_index=True))

bus_2010

In [ ]:
# 'P11_002'列で4のみを含む行を削除する関数
def remove_only_fours(df):
    # 'P11_002'列の値が4のみを含むかチェックする正規表現パターン
    pattern = r'^4+$'

    # 条件に合わない行を保持（つまり、4のみの行を削除）
    filtered_df = bus_2010[~bus_2010['P11_002'].astype(str).str.match(pattern)]

    return filtered_df

# データフレームにこの関数を適用
bus_2010_delete4 = remove_only_fours(bus_2010)
bus_2010_delete4

In [ ]:
bus_2010_delete4_unique = bus_2010_delete4.drop_duplicates(subset=['P11_001', 'geometry'], keep='first')
bus_2010_delete4_unique

In [ ]:
# 日本に適したカスタム等積投影法を定義（人口計算コードと同じ）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# バス停データをGeoDataFrameに変換し、CRSをWGS84に設定（既に設定されている場合は不要）
bus_2022_2023_gdf_unique = gpd.GeoDataFrame(bus_2022_2023_gdf_unique, geometry='geometry', crs='EPSG:6668')

# バス停データをカスタム等積投影法に変換
bus_2022_2023_gdf_unique = bus_2022_2023_gdf_unique.to_crs(custom_equal_area_crs)

# バス停と駅のバッファの空間結合を実施
bus_stops_within_buffers = gpd.sjoin(bus_2022_2023_gdf_unique, buffers, how='inner', predicate='within')

# 各駅ごとにバス停数をカウント
bus_stop_counts = bus_stops_within_buffers.groupby('station_id').size().reset_index(name='bus_stops_within_2km_2022')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# バス停と駅の1kmバッファの空間結合を実施
bus_stops_within_1km_buffers = gpd.sjoin(bus_2022_2023_gdf_unique, buffers_1km, how='inner', predicate='within')

# 各駅ごとに1km圏内のバス停数をカウント
bus_stop_counts_1km = bus_stops_within_1km_buffers.groupby('station_id').size().reset_index(name='bus_stops_within_1km_2022')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts_1km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# バス停と駅の5kmバッファの空間結合を実施
bus_stops_within_5km_buffers = gpd.sjoin(bus_2022_2023_gdf_unique, buffers_5km, how='inner', predicate='within')

# 各駅ごとに5km圏内のバス停数をカウント
bus_stop_counts_5km = bus_stops_within_5km_buffers.groupby('station_id').size().reset_index(name='bus_stops_within_5km_2022')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts_5km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# バス停と駅の10kmバッファの空間結合を実施
bus_stops_within_10km_buffers = gpd.sjoin(bus_2022_2023_gdf_unique, buffers_10km, how='inner', predicate='within')

# 各駅ごとに10km圏内のバス停数をカウント
bus_stop_counts_10km = bus_stops_within_10km_buffers.groupby('station_id').size().reset_index(name='bus_stops_within_10km_2022')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts_10km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 日本に適したカスタム等積投影法を定義（人口計算コードと同じ）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# バス停データをGeoDataFrameに変換し、CRSをWGS84に設定（既に設定されている場合は不要）
bus_2010_delete4_unique = gpd.GeoDataFrame(bus_2010_delete4_unique, geometry='geometry', crs='EPSG:6668')

# バス停データをカスタム等積投影法に変換
bus_2010_delete4_unique = bus_2010_delete4_unique.to_crs(custom_equal_area_crs)

# バス停と駅のバッファの空間結合を実施
bus_stops_within_buffers = gpd.sjoin(bus_2010_delete4_unique, buffers, how='inner', predicate='within')

# 各駅ごとにバス停数をカウント
bus_stop_counts = bus_stops_within_buffers.groupby('station_id').size().reset_index(name='bus_stops_within_2km_2010')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 2010年バス停データと1kmバッファの空間結合を実施
bus_stops_within_1km_buffers_2010 = gpd.sjoin(bus_2010_delete4_unique, buffers_1km, how='inner', predicate='within')

# 各駅ごとに1km圏内のバス停数をカウント
bus_stop_counts_1km_2010 = bus_stops_within_1km_buffers_2010.groupby('station_id').size().reset_index(name='bus_stops_within_1km_2010')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts_1km_2010, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# バス停と駅の5kmバッファの空間結合を実施
bus_stops_within_5km_buffers_2010 = gpd.sjoin(bus_2010_delete4_unique, buffers_5km, how='inner', predicate='within')

# 各駅ごとに5km圏内のバス停数をカウント
bus_stop_counts_5km_2010 = bus_stops_within_5km_buffers_2010.groupby('station_id').size().reset_index(name='bus_stops_within_5km_2010')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts_5km_2010, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 2010年バス停データと10kmバッファの空間結合を実施
bus_stops_within_10km_buffers_2010 = gpd.sjoin(bus_2010_delete4_unique, buffers_10km, how='inner', predicate='within')

# 各駅ごとに10km圏内のバス停数をカウント
bus_stop_counts_10km_2010 = bus_stops_within_10km_buffers_2010.groupby('station_id').size().reset_index(name='bus_stops_within_10km_2010')

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(bus_stop_counts_10km_2010, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
nan_counts = df_merged.isna().sum()

print(nan_counts)

In [ ]:
df_merged['bus_stops_within_2km_2022'] = df_merged['bus_stops_within_2km_2022'].fillna(0)
df_merged['bus_stops_within_1km_2022'] = df_merged['bus_stops_within_1km_2022'].fillna(0)
df_merged['bus_stops_within_5km_2022'] = df_merged['bus_stops_within_5km_2022'].fillna(0)
df_merged['bus_stops_within_10km_2022'] = df_merged['bus_stops_within_10km_2022'].fillna(0)

df_merged['bus_stops_within_2km_2010'] = df_merged['bus_stops_within_2km_2010'].fillna(0)
df_merged['bus_stops_within_1km_2010'] = df_merged['bus_stops_within_1km_2010'].fillna(0)
df_merged['bus_stops_within_5km_2010'] = df_merged['bus_stops_within_5km_2010'].fillna(0)
df_merged['bus_stops_within_10km_2010'] = df_merged['bus_stops_within_10km_2010'].fillna(0)

（７）事業所数・就業者数

In [ ]:
def load_economic_census_2021_data(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*.txt"))
    df_list = []
    for file in all_files:
        df = pd.read_csv(
            file,
            header=0,              # 最初のヘッダー行のみを使用
            skiprows=[1],          # 2行目のヘッダーをスキップ
            encoding='cp932',      # 日本語のエンコーディング
            na_values=['*', '']    # 欠損値の扱い
        )
        df_list.append(df)
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

# フォルダのパスを指定
folder_path = 'economic_census_2021_1km'

# データフレームを取得
df_economic_census_2021 = load_economic_census_2021_data(folder_path)

df_economic_census_2021

In [ ]:
# データフレームを横結合する
df_economic_census_2021 = df_economic_census_2021.merge(df_boundary[['KEY_CODE', 'geometry']],
                               on='KEY_CODE',
                               how='left')
df_economic_census_2021

In [ ]:
# 日本に適したカスタム等積投影法を定義（修正済み）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# df_economic_census_2021をGeoDataFrameに変換し、CRSを設定
df_economic_census_2021 = gpd.GeoDataFrame(df_economic_census_2021, geometry='geometry', crs='EPSG:6668')

# 面積計算とバッファ作成のために、両方のデータフレームをカスタム等積投影法に変換
df_economic_census_2021_equal_area = df_economic_census_2021.to_crs(custom_equal_area_crs)
buffers_equal_area = buffers.to_crs(custom_equal_area_crs)

# 経済センサスポリゴンの面積を計算
df_economic_census_2021_equal_area['original_area'] = df_economic_census_2021_equal_area.area

# バッファと経済センサスポリゴンの交差部分を計算
intersection = gpd.overlay(
    df_economic_census_2021_equal_area,
    buffers_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection['intersection_area'] = intersection.area

# 面積比を計算
intersection['area_ratio'] = intersection['intersection_area'] / intersection['original_area']

# 面積比を用いて事業所数と就業者数を按分
intersection['weighted_establishments'] = intersection['T001146001'] * intersection['area_ratio']
intersection['weighted_employees'] = intersection['T001146022'] * intersection['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station = intersection.groupby('station_id')[['weighted_establishments', 'weighted_employees']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station = establishments_and_employees_by_station.rename(columns={
    'weighted_establishments': 'establishments_within_2km_2021',
    'weighted_employees': 'employees_within_2km_2021'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station = all_station_ids.merge(establishments_and_employees_by_station, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 1kmバッファをカスタム等積投影法に変換（buffers_1kmが存在し、EPSG:6668である前提）
buffers_1km_equal_area = buffers_1km.to_crs(custom_equal_area_crs)

# バッファと経済センサスポリゴン（2021年）の交差部分を計算（1kmバッファ）
intersection_1km = gpd.overlay(
    df_economic_census_2021_equal_area,
    buffers_1km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム（overlayの結果によってはstation_id_1またはstation_id_2となる場合があります）
if 'station_id_1' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_1km['intersection_area'] = intersection_1km.area

# 面積比を計算
intersection_1km['area_ratio'] = intersection_1km['intersection_area'] / intersection_1km['original_area']

# 面積比を用いて事業所数と就業者数を按分
intersection_1km['weighted_establishments_1km'] = intersection_1km['T001146001'] * intersection_1km['area_ratio']
intersection_1km['weighted_employees_1km'] = intersection_1km['T001146022'] * intersection_1km['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station_1km = intersection_1km.groupby('station_id')[['weighted_establishments_1km', 'weighted_employees_1km']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station_1km = establishments_and_employees_by_station_1km.rename(columns={
    'weighted_establishments_1km': 'establishments_within_1km_2021',
    'weighted_employees_1km': 'employees_within_1km_2021'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids_1km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station_1km = all_station_ids_1km.merge(establishments_and_employees_by_station_1km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station_1km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 5kmバッファをカスタム等積投影法に変換（buffers_5kmが存在し、EPSG:6668である前提）
buffers_5km_equal_area = buffers_5km.to_crs(custom_equal_area_crs)

# バッファと経済センサスポリゴン（2021年）の交差部分を計算（5kmバッファ）
intersection_5km = gpd.overlay(
    df_economic_census_2021_equal_area,
    buffers_5km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_5km['intersection_area'] = intersection_5km.area

# 面積比を計算
intersection_5km['area_ratio'] = intersection_5km['intersection_area'] / intersection_5km['original_area']

# 面積比を用いて事業所数と就業者数を按分
intersection_5km['weighted_establishments_5km'] = intersection_5km['T001146001'] * intersection_5km['area_ratio']
intersection_5km['weighted_employees_5km'] = intersection_5km['T001146022'] * intersection_5km['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station_5km = intersection_5km.groupby('station_id')[['weighted_establishments_5km', 'weighted_employees_5km']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station_5km = establishments_and_employees_by_station_5km.rename(columns={
    'weighted_establishments_5km': 'establishments_within_5km_2021',
    'weighted_employees_5km': 'employees_within_5km_2021'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids_5km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station_5km = all_station_ids_5km.merge(establishments_and_employees_by_station_5km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station_5km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 10kmバッファをカスタム等積投影法に変換（buffers_10kmが存在し、EPSG:6668である前提）
buffers_10km_equal_area = buffers_10km.to_crs(custom_equal_area_crs)

# バッファと経済センサスポリゴン（2021年）の交差部分を計算（10kmバッファ）
intersection_10km = gpd.overlay(
    df_economic_census_2021_equal_area,
    buffers_10km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_10km['intersection_area'] = intersection_10km.area

# 面積比を計算
intersection_10km['area_ratio'] = intersection_10km['intersection_area'] / intersection_10km['original_area']

# 面積比を用いて事業所数と就業者数を按分
intersection_10km['weighted_establishments_10km'] = intersection_10km['T001146001'] * intersection_10km['area_ratio']
intersection_10km['weighted_employees_10km'] = intersection_10km['T001146022'] * intersection_10km['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station_10km = intersection_10km.groupby('station_id')[['weighted_establishments_10km', 'weighted_employees_10km']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station_10km = establishments_and_employees_by_station_10km.rename(columns={
    'weighted_establishments_10km': 'establishments_within_10km_2021',
    'weighted_employees_10km': 'employees_within_10km_2021'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids_10km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station_10km = all_station_ids_10km.merge(establishments_and_employees_by_station_10km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station_10km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
def load_economic_census_2016_data(folder_path):
    all_files = glob.glob(os.path.join(folder_path, "*.txt"))
    df_list = []
    for file in all_files:
        df = pd.read_csv(
            file,
            header=0,              # 最初のヘッダー行のみを使用
            skiprows=[1],          # 2行目のヘッダーをスキップ
            encoding='cp932',      # 日本語のエンコーディング
            na_values=['*', '']    # 欠損値の扱い
        )
        df_list.append(df)
    full_df = pd.concat(df_list, ignore_index=True)
    return full_df

# フォルダのパスを指定
folder_path = 'economic_census_2016_1km'

# データフレームを取得
df_economic_census_2016 = load_economic_census_2016_data(folder_path)

df_economic_census_2016

In [ ]:
# データフレームを横結合する
df_economic_census_2016 = df_economic_census_2016.merge(df_boundary[['KEY_CODE', 'geometry']],
                               on='KEY_CODE',
                               how='left')
df_economic_census_2016

In [ ]:
# 日本に適したカスタム等積投影法を定義（修正済み）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# df_economic_census_2016をGeoDataFrameに変換し、CRSを設定
df_economic_census_2016 = gpd.GeoDataFrame(df_economic_census_2016, geometry='geometry', crs='EPSG:6668')

# 面積計算とバッファ作成のために、両方のデータフレームをカスタム等積投影法に変換
df_economic_census_2016_equal_area = df_economic_census_2016.to_crs(custom_equal_area_crs)
buffers_equal_area = buffers.to_crs(custom_equal_area_crs)

# 経済センサスポリゴンの面積を計算
df_economic_census_2016_equal_area['original_area'] = df_economic_census_2016_equal_area.area

# バッファと経済センサスポリゴンの交差部分を計算
intersection = gpd.overlay(
    df_economic_census_2016_equal_area,
    buffers_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection['intersection_area'] = intersection.area

# 面積比を計算
intersection['area_ratio'] = intersection['intersection_area'] / intersection['original_area']

# 面積比を用いて事業所数と就業者数を按分
intersection['weighted_establishments'] = intersection['T000917001'] * intersection['area_ratio']
intersection['weighted_employees'] = intersection['T000917020'] * intersection['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station = intersection.groupby('station_id')[['weighted_establishments', 'weighted_employees']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station = establishments_and_employees_by_station.rename(columns={
    'weighted_establishments': 'establishments_within_2km_2016',
    'weighted_employees': 'employees_within_2km_2016'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station = all_station_ids.merge(establishments_and_employees_by_station, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 1kmバッファをカスタム等積投影法に変換（buffers_1kmが存在し、EPSG:6668である前提）
buffers_1km_equal_area = buffers_1km.to_crs(custom_equal_area_crs)

# バッファと2016年経済センサスポリゴンの交差部分を計算（1kmバッファ）
intersection_1km = gpd.overlay(
    df_economic_census_2016_equal_area,
    buffers_1km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム（overlayの結果、station_id_1またはstation_id_2となる可能性があるため）
if 'station_id_1' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_1km['intersection_area'] = intersection_1km.area

# 面積比を計算
intersection_1km['area_ratio'] = intersection_1km['intersection_area'] / intersection_1km['original_area']

# 面積比を用いて事業所数と就業者数を按分（2016年の該当カラムを使用）
intersection_1km['weighted_establishments_1km_2016'] = intersection_1km['T000917001'] * intersection_1km['area_ratio']
intersection_1km['weighted_employees_1km_2016'] = intersection_1km['T000917020'] * intersection_1km['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station_1km = intersection_1km.groupby('station_id')[['weighted_establishments_1km_2016', 'weighted_employees_1km_2016']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station_1km = establishments_and_employees_by_station_1km.rename(columns={
    'weighted_establishments_1km_2016': 'establishments_within_1km_2016',
    'weighted_employees_1km_2016': 'employees_within_1km_2016'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids_1km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station_1km = all_station_ids_1km.merge(establishments_and_employees_by_station_1km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station_1km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 5kmバッファをカスタム等積投影法に変換（buffers_5kmが存在し、EPSG:6668である前提）
buffers_5km_equal_area = buffers_5km.to_crs(custom_equal_area_crs)

# バッファと2016年経済センサスポリゴンの交差部分を計算（5kmバッファ）
intersection_5km = gpd.overlay(
    df_economic_census_2016_equal_area,
    buffers_5km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム（overlayの結果、station_id_1またはstation_id_2となる可能性があるため）
if 'station_id_1' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_5km['intersection_area'] = intersection_5km.area

# 面積比を計算
intersection_5km['area_ratio'] = intersection_5km['intersection_area'] / intersection_5km['original_area']

# 面積比を用いて事業所数と就業者数を按分（2016年の該当カラムを使用）
intersection_5km['weighted_establishments_5km_2016'] = intersection_5km['T000917001'] * intersection_5km['area_ratio']
intersection_5km['weighted_employees_5km_2016'] = intersection_5km['T000917020'] * intersection_5km['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station_5km = intersection_5km.groupby('station_id')[['weighted_establishments_5km_2016', 'weighted_employees_5km_2016']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station_5km = establishments_and_employees_by_station_5km.rename(columns={
    'weighted_establishments_5km_2016': 'establishments_within_5km_2016',
    'weighted_employees_5km_2016': 'employees_within_5km_2016'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids_5km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station_5km = all_station_ids_5km.merge(establishments_and_employees_by_station_5km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station_5km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 10kmバッファをカスタム等積投影法に変換（buffers_10kmが存在し、EPSG:6668である前提）
buffers_10km_equal_area = buffers_10km.to_crs(custom_equal_area_crs)

# バッファと2016年経済センサスポリゴンの交差部分を計算（10kmバッファ）
intersection_10km = gpd.overlay(
    df_economic_census_2016_equal_area,
    buffers_10km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム（overlay結果によってはstation_id_1またはstation_id_2になる可能性あり）
if 'station_id_1' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_10km['intersection_area'] = intersection_10km.area

# 面積比を計算
intersection_10km['area_ratio'] = intersection_10km['intersection_area'] / intersection_10km['original_area']

# 面積比を用いて事業所数と就業者数を按分（2016年の該当カラム T000917001: 事業所数、T000917020: 就業者数）
intersection_10km['weighted_establishments_10km_2016'] = intersection_10km['T000917001'] * intersection_10km['area_ratio']
intersection_10km['weighted_employees_10km_2016'] = intersection_10km['T000917020'] * intersection_10km['area_ratio']

# 各駅ごとに按分した事業所数と就業者数を合計
establishments_and_employees_by_station_10km = intersection_10km.groupby('station_id')[['weighted_establishments_10km_2016', 'weighted_employees_10km_2016']].sum().reset_index()

# 列名を適切にリネーム
establishments_and_employees_by_station_10km = establishments_and_employees_by_station_10km.rename(columns={
    'weighted_establishments_10km_2016': 'establishments_within_10km_2016',
    'weighted_employees_10km_2016': 'employees_within_10km_2016'
})

# 全ての駅IDを含むデータフレームを作成
all_station_ids_10km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと計算結果をマージし、NaNを0で埋める
establishments_and_employees_by_station_10km = all_station_ids_10km.merge(establishments_and_employees_by_station_10km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(establishments_and_employees_by_station_10km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
nan_counts = df_merged.isna().sum()

print(nan_counts)

（８）人口推計データ

In [ ]:
#人口推計データ
# 人口推計データのフォルダのパス
folder_path = 'estimated_future_population_2018'

# フォルダ内のすべての.shpファイルを取得
shapefiles = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.shp')]

# すべてのシェープファイルを読み込み、リストに格納
gdf_list = [gpd.read_file(shp) for shp in shapefiles]

# すべてのGeoDataFrameを一つのGeoDataFrameに結合
df_estimated_population = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True))

# 結果のGeoDataFrameを表示
df_estimated_population

In [ ]:
# 日本に適したカスタム等積投影法を定義（修正済み）
custom_equal_area_crs = {
    'proj': 'aea',
    'lat_1': 29.5,    # 標準緯線1
    'lat_2': 45.5,    # 標準緯線2
    'lat_0': 35,      # 原点の緯度
    'lon_0': 135,     # 原点の経度
    'x_0': 0,
    'y_0': 0,
    'ellps': 'GRS80',
    'units': 'm'
}

# df_estimated_populationをGeoDataFrameに変換し、CRSをJGD2011に設定
df_estimated_population = gpd.GeoDataFrame(df_estimated_population, geometry='geometry', crs='EPSG:6668')

# 将来人口ポリゴンをカスタム等積投影法に変換
df_estimated_population_equal_area = df_estimated_population.to_crs(custom_equal_area_crs)

# 将来人口ポリゴンの面積を計算
df_estimated_population_equal_area['original_area'] = df_estimated_population_equal_area.area

# バッファと将来人口ポリゴンの交差部分を計算
intersection = gpd.overlay(
    df_estimated_population_equal_area,
    buffers_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection.columns:
    intersection = intersection.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection['intersection_area'] = intersection.area

# 面積比を計算
intersection['area_ratio'] = intersection['intersection_area'] / intersection['original_area']

# 各年の人口推計列を指定
population_columns = ['PTN_2020', 'PTN_2025', 'PTN_2030', 'PTN_2035', 'PTN_2040', 'PTN_2045', 'PTN_2050']

# 各年の人口推計を按分
for col in population_columns:
    year = col[-4:]
    intersection[f'estimated_population_within_2km_{year}'] = intersection[col] * intersection['area_ratio']

# 各駅ごとに按分した人口を合計
population_columns_within_2km = [f'estimated_population_within_2km_{col[-4:]}' for col in population_columns]
population_by_station = intersection.groupby('station_id')[population_columns_within_2km].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと人口データをマージし、NaNを0で埋める
population_by_station = all_station_ids.merge(population_by_station, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 1kmバッファをカスタム等積投影法に変換
buffers_1km_equal_area = buffers_1km.to_crs(custom_equal_area_crs)

# バッファと将来人口ポリゴンの交差部分を計算（1kmバッファ）
intersection_1km = gpd.overlay(
    df_estimated_population_equal_area,
    buffers_1km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム（overlayの結果でstation_id_1またはstation_id_2となる可能性がある）
if 'station_id_1' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_1km.columns:
    intersection_1km = intersection_1km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_1km['intersection_area'] = intersection_1km.area

# 面積比を計算
intersection_1km['area_ratio'] = intersection_1km['intersection_area'] / intersection_1km['original_area']

# 各年の人口推計列を指定（2km計算コードと同様）
population_columns = ['PTN_2020', 'PTN_2025', 'PTN_2030', 'PTN_2035', 'PTN_2040', 'PTN_2045', 'PTN_2050']

# 各年の人口推計を按分（1km圏内用）
for col in population_columns:
    year = col[-4:]
    intersection_1km[f'estimated_population_within_1km_{year}'] = intersection_1km[col] * intersection_1km['area_ratio']

# 各駅ごとに按分した人口を合計
population_columns_within_1km = [f'estimated_population_within_1km_{col[-4:]}' for col in population_columns]
population_by_station_1km = intersection_1km.groupby('station_id')[population_columns_within_1km].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_1km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと人口データをマージし、NaNを0で埋める
population_by_station_1km = all_station_ids_1km.merge(population_by_station_1km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_1km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 5kmバッファをカスタム等積投影法に変換
buffers_5km_equal_area = buffers_5km.to_crs(custom_equal_area_crs)

# バッファと将来人口ポリゴンの交差部分を計算（5kmバッファ）
intersection_5km = gpd.overlay(
    df_estimated_population_equal_area,
    buffers_5km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_5km.columns:
    intersection_5km = intersection_5km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_5km['intersection_area'] = intersection_5km.area

# 面積比を計算
intersection_5km['area_ratio'] = intersection_5km['intersection_area'] / intersection_5km['original_area']

# 各年の人口推計列を指定（これまでと同様）
population_columns = ['PTN_2020', 'PTN_2025', 'PTN_2030', 'PTN_2035', 'PTN_2040', 'PTN_2045', 'PTN_2050']

# 各年の人口推計を按分（5km圏内用）
for col in population_columns:
    year = col[-4:]
    intersection_5km[f'estimated_population_within_5km_{year}'] = intersection_5km[col] * intersection_5km['area_ratio']

# 各駅ごとに按分した人口を合計
population_columns_within_5km = [f'estimated_population_within_5km_{col[-4:]}' for col in population_columns]
population_by_station_5km = intersection_5km.groupby('station_id')[population_columns_within_5km].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_5km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと人口データをマージし、NaNを0で埋める
population_by_station_5km = all_station_ids_5km.merge(population_by_station_5km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_5km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
# 10kmバッファをカスタム等積投影法に変換
buffers_10km_equal_area = buffers_10km.to_crs(custom_equal_area_crs)

# バッファと将来人口ポリゴンの交差部分を計算（10kmバッファ）
intersection_10km = gpd.overlay(
    df_estimated_population_equal_area,
    buffers_10km_equal_area,
    how='intersection'
)

# 列名を確認して'station_id'をリネーム
if 'station_id_1' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_1': 'station_id'})
elif 'station_id_2' in intersection_10km.columns:
    intersection_10km = intersection_10km.rename(columns={'station_id_2': 'station_id'})

# 交差部分の面積を計算
intersection_10km['intersection_area'] = intersection_10km.area

# 面積比を計算
intersection_10km['area_ratio'] = intersection_10km['intersection_area'] / intersection_10km['original_area']

# 各年の人口推計列を指定
population_columns = ['PTN_2020', 'PTN_2025', 'PTN_2030', 'PTN_2035', 'PTN_2040', 'PTN_2045', 'PTN_2050']

# 各年の人口推計を按分（10km圏内用）
for col in population_columns:
    year = col[-4:]
    intersection_10km[f'estimated_population_within_10km_{year}'] = intersection_10km[col] * intersection_10km['area_ratio']

# 各駅ごとに按分した人口を合計
population_columns_within_10km = [f'estimated_population_within_10km_{col[-4:]}' for col in population_columns]
population_by_station_10km = intersection_10km.groupby('station_id')[population_columns_within_10km].sum().reset_index()

# 全ての駅IDを含むデータフレームを作成
all_station_ids_10km = df_merged[['station_id']].drop_duplicates()

# 全ての駅IDと人口データをマージし、NaNを0で埋める
population_by_station_10km = all_station_ids_10km.merge(population_by_station_10km, on='station_id', how='left').fillna(0)

# 結果をdf_mergedにマージ
df_merged = df_merged.merge(population_by_station_10km, on='station_id', how='left')

# 結果を表示
df_merged

In [ ]:
nan_counts = df_merged.isna().sum()

print(nan_counts)

In [ ]:
import matplotlib.pyplot as plt
# 予測誤差を計算（予測/実績）
df_merged['error_population_within_2km_2020'] = df_merged['estimated_population_within_2km_2020'] / df_merged['population_within_2km_2020']
df_merged['error_population_within_1km_2020'] = df_merged['estimated_population_within_1km_2020'] / df_merged['population_within_1km_2020']
df_merged['error_population_within_5km_2020'] = df_merged['estimated_population_within_5km_2020'] / df_merged['population_within_5km_2020']
df_merged['error_population_within_10km_2020'] = df_merged['estimated_population_within_10km_2020'] / df_merged['population_within_10km_2020']

# ヒストグラムの作成（5%刻み、範囲を0.5から1.5に設定）
bin_edges = [i * 0.05 for i in range(10, 31)]  # 0.5から1.5までを5%刻み
plt.hist(df_merged['error_population_within_2km_2020'], bins=bin_edges, edgecolor='black')
plt.xlim(0.5, 1.5)
plt.xlabel('Prediction Error (Predicted / Actual)')
plt.ylabel('Frequency')
plt.title('Histogram of Prediction Error (5% bins, range 0.5 to 1.5)')
plt.show()

# ±20%以内の件数をカウント
within_20_percent = df_merged[(df_merged['error_population_within_2km_2020'] >= 0.8) & (df_merged['error_population_within_2km_2020'] <= 1.2)].shape[0]
within_10_percent = df_merged[(df_merged['error_population_within_2km_2020'] >= 0.9) & (df_merged['error_population_within_2km_2020'] <= 1.1)].shape[0]
within_5_percent = df_merged[(df_merged['error_population_within_2km_2020'] >= 0.95) & (df_merged['error_population_within_2km_2020'] <= 1.05)].shape[0]

# 全体の件数に対する割合を計算
total_count = df_merged.shape[0]
within_20_percent_ratio = within_20_percent / total_count * 100
within_10_percent_ratio = within_10_percent / total_count * 100
within_5_percent_ratio = within_5_percent / total_count * 100

print(f"±20%以内の件数の割合: {within_20_percent_ratio:.2f}%")
print(f"±10%以内の件数の割合: {within_10_percent_ratio:.2f}%")
print(f"±5%以内の件数の割合: {within_5_percent_ratio:.2f}%")

In [ ]:
# データフレーム df の特定の列 'column_name' にある NaN の個数を確認
nan_count = df_merged['passengers_2022'].isna().sum()

print(f"NaNの個数: {nan_count}")

In [ ]:

# NaNが含まれる行を抽出
nan_rows = df_merged[df_merged['connect_line_number'].isna()].reset_index(drop=True)
nan_rows

増減率の前提条件を合わせるための計算

In [ ]:
# 条件と処理のリスト
adjustments = [
    {'station': '町田', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -254478},
    {'station': '羽田空港第３ターミナル', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -32528},
    {'station': '羽田空港第１・第２ターミナル', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -101471},
    {'station': '代々木上原', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -257017},
    {'station': '北千住', 'pref': '東京都', 'column': 'passengers_2021', 'adjustment': -351035},
    {'station': '北千住', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -428570},
    {'station': '北千住', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -451541},
    {'station': '亀戸', 'pref': '東京都', 'column': 'passengers_2021', 'adjustment': -22000},
    {'station': '亀戸', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -23651},
    {'station': '亀戸', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -24660},
    {'station': '天空橋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -18262},
    {'station': '南千住', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -5672},
    {'station': '南千住', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -6249},
    {'station': '品川', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -508137}, # 前は-233916
    {'station': '池袋', 'pref': '東京都', 'column': 'passengers_2021', 'adjustment': -351651},
    {'station': '池袋', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -388238},
    {'station': '池袋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -408382},
    {'station': '大崎', 'pref': '東京都', 'column': 'passengers_2021', 'adjustment': -39469},
    {'station': '大崎', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -45860},
    {'station': '大崎', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -53416},
    {'station': '市ケ谷', 'pref': '東京都', 'column': 'passengers_2020', 'adjustment': -66889},
    {'station': '市ケ谷', 'pref': '東京都', 'column': 'passengers_2021', 'adjustment': -70806},
    {'station': '市ケ谷', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -78112},
    {'station': '分倍河原', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -20296},
    {'station': '分倍河原', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -83916},
    {'station': '高幡不動', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -47988},
    {'station': '高幡不動', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -49257},
    {'station': '多摩動物公園', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -4207},
    {'station': '多摩動物公園', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -4108},
    {'station': '吉祥寺', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -122831},
    {'station': '吉祥寺', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -126063},
    {'station': '高尾', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -22376},
    {'station': '高尾', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -23272},
    {'station': '新宿', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -613639},
    {'station': '新宿', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -1094332},
    {'station': '神保町', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -81664},
    {'station': '新御徒町', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -19160},
    {'station': '新御徒町', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -21433},
    {'station': '中延', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -20487},
    {'station': '巣鴨', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -65775},
    {'station': '日暮里', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -102143},
    {'station': '月島', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -67069},
    {'station': '東銀座', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -76513},
    {'station': '日比谷', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -90430},
    {'station': '浅草橋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -48054},
    {'station': '天王洲アイル', 'pref': '東京都', 'column': 'passengers_2021', 'adjustment': -11283},
    {'station': '天王洲アイル', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -13226},
    {'station': '天王洲アイル', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -14895},
    {'station': '両国', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -34415},
    {'station': '六本木', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -105196},
    {'station': '東新宿', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -39476},
    {'station': '人形町', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -65602},
    {'station': '門前仲町', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -101848},
    {'station': '青山一丁目', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -93509},
    {'station': '麻布十番', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -41990},
    {'station': '住吉', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -52392},
    {'station': '代々木', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -54704},
    {'station': '白金高輪', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -35889},
    {'station': '中井', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -26686},
    {'station': '日本橋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -163127},
    {'station': '清澄白河', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -54925},
    {'station': '水道橋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -68577},
    {'station': '九段下', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -153242},
    {'station': '練馬', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -116257},
    {'station': '東中野', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -35805},
    {'station': '白金台', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -16112},
    {'station': '豊島園', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -14849},
    {'station': '新橋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -413487},
    {'station': '新宿三丁目', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -142867},
    {'station': '中野坂上', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -70705},
    {'station': '本郷三丁目', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -49496},
    {'station': '五反田', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -200686},
    {'station': '東京', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -590084},
    {'station': '飯田橋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -216805},
    {'station': '目黒', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -447176},
    {'station': '町屋', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -59032},
    {'station': '泉岳寺', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -177838},
    {'station': '渋谷', 'pref': '東京都', 'column': 'passengers_2022', 'adjustment': -274505},
    {'station': '渋谷', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -285759},
    {'station': '西日暮里', 'pref': '東京都', 'column': 'passengers_2023', 'adjustment': -239178},

    {'station': '海老名', 'pref': '神奈川県', 'column': 'passengers_2021', 'adjustment': -89216},
    {'station': '海老名', 'pref': '神奈川県', 'column': 'passengers_2022', 'adjustment': -98040},
    {'station': '海老名', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -239940},
    {'station': '八丁畷', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -15874},
    {'station': '厚木', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -18993},
    {'station': '羽沢横浜国大', 'pref': '神奈川県', 'column': 'passengers_2021', 'adjustment': -24655},
    {'station': '羽沢横浜国大', 'pref': '神奈川県', 'column': 'passengers_2022', 'adjustment': -29336},
    {'station': '羽沢横浜国大', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -31622},
    {'station': '金沢八景', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -57282},
    {'station': '湘南台', 'pref': '神奈川県', 'column': 'passengers_2021', 'adjustment': -22355},
    {'station': '湘南台', 'pref': '神奈川県', 'column': 'passengers_2022', 'adjustment': -24105},
    {'station': '湘南台', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -112904},
    {'station': '登戸', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -155310},
    {'station': '上大岡', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -126210},
    {'station': '弘明寺', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -26632},
    {'station': '藤沢', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -257010},
    {'station': '中央林間', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -90052},
    {'station': '横浜', 'pref': '神奈川県', 'column': 'passengers_2021', 'adjustment': -305183},
    {'station': '横浜', 'pref': '神奈川県', 'column': 'passengers_2022', 'adjustment': -329228},
    {'station': '横浜', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -607822},
    {'station': '橋本', 'pref': '神奈川県', 'column': 'passengers_2022', 'adjustment': -56999},
    {'station': '橋本', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -59596},
    {'station': '新杉田', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -33410},
    {'station': '大船', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -88996},
    {'station': '鎌倉', 'pref': '神奈川県', 'column': 'passengers_2023', 'adjustment': -39593},

    {'station': '南流山', 'pref': '千葉県', 'column': 'passengers_2022', 'adjustment': -34909},
    {'station': '南流山', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -37577},
    {'station': '柏', 'pref': '千葉県', 'column': 'passengers_2021', 'adjustment': -123592},
    {'station': '柏', 'pref': '千葉県', 'column': 'passengers_2022', 'adjustment': -135064},
    {'station': '柏', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -141087},
    {'station': '船橋', 'pref': '千葉県', 'column': 'passengers_2021', 'adjustment': -98586},
    {'station': '船橋', 'pref': '千葉県', 'column': 'passengers_2022', 'adjustment': -107773},
    {'station': '船橋', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -112702},
    {'station': '成田空港', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -6449},
    {'station': '空港第２ビル', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -5267},
    {'station': '本八幡', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -53952},
    {'station': '松戸', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -88200},
    {'station': '千葉みなと', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -16602},
    {'station': '都賀', 'pref': '千葉県', 'column': 'passengers_2023', 'adjustment': -19376},

    {'station': '小川町', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -7424},
    {'station': '小川町', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -7759},
    {'station': '小川町', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -8394},
    {'station': '川越', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -99262},
    {'station': '川越', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -110102},
    {'station': '川越', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -114751},
    {'station': '久喜', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -41047},
    {'station': '久喜', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -44527},
    {'station': '久喜', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -46131},
    {'station': '羽生', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -10589},
    {'station': '羽生', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -11399},
    {'station': '羽生', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -11656},
    {'station': '和光市', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -138207},
    {'station': '和光市', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -152949},
    {'station': '和光市', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -162203},
    {'station': '栗橋', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -9015},
    {'station': '栗橋', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -9730},
    {'station': '栗橋', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -10131},
    {'station': '大宮', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -110181},
    {'station': '大宮', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -118886},
    {'station': '大宮', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -368951}, #前は-124558
    {'station': '寄居', 'pref': '埼玉県', 'column': 'passengers_2021', 'adjustment': -2928},
    {'station': '寄居', 'pref': '埼玉県', 'column': 'passengers_2022', 'adjustment': -3323},
    {'station': '寄居', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -3748},
    {'station': '東川口', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -36752},
    {'station': '熊谷', 'pref': '埼玉県', 'column': 'passengers_2023', 'adjustment': -26991}
]

# 条件に基づいて値を調整
for adj in adjustments:
    df_merged.loc[(df_merged['jan_station'] == adj['station']) & (df_merged['pref_name'] == adj['pref']), adj['column']] += adj['adjustment']

In [ ]:
# 新整備場駅の2020年の人口を0にする
# df_merged.loc[(df_merged['jan_station'] == '新整備場') & (df_merged['pref_name'] == '東京都'), 'population_within_2km_2020'] = 0

（９）増減率

In [ ]:
# 列名のリストを作成
columns = [
    'passengers_2000', 'passengers_2001', 'passengers_2002', 'passengers_2003', 'passengers_2004',
    'passengers_2005', 'passengers_2006', 'passengers_2007', 'passengers_2008', 'passengers_2009',
    'passengers_2010', 'passengers_2011', 'passengers_2012', 'passengers_2013', 'passengers_2014',
    'passengers_2015', 'passengers_2016', 'passengers_2017', 'passengers_2018', 'passengers_2019',
    'passengers_2020', 'passengers_2021', 'passengers_2022', 'passengers_2023'
]

# 0の値をNaNに変換
df_merged[columns] = df_merged[columns].replace(0, np.nan)

In [ ]:
# 列名のリストを作成
columns = [
    'passengers_2000', 'passengers_2001', 'passengers_2002', 'passengers_2003', 'passengers_2004',
    'passengers_2005', 'passengers_2006', 'passengers_2007', 'passengers_2008', 'passengers_2009',
    'passengers_2010', 'passengers_2011', 'passengers_2012', 'passengers_2013', 'passengers_2014',
    'passengers_2015', 'passengers_2016', 'passengers_2017', 'passengers_2018', 'passengers_2019',
    'passengers_2020', 'passengers_2021', 'passengers_2022', 'passengers_2023'
]

# 指定列のすべてがNaNの場合、その行を削除
df_merged = df_merged.dropna(subset=columns, how='all')

In [ ]:
# 乗降者数の増減率
# passengers_beforeの優先順位リストを定義
cols_before_passengers = [
    'passengers_2019', 'passengers_2018', 'passengers_2017',
    'passengers_2016', 'passengers_2015', 'passengers_2014',
    'passengers_2013', 'passengers_2012', 'passengers_2011'
]

# passengers_before列を作成
df_merged['passengers_before'] = df_merged[cols_before_passengers].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_passengers = [
    'passengers_2023', 'passengers_2022',
    'passengers_2021', 'passengers_2020'
]

# passengers_after列を作成
df_merged['passengers_after'] = df_merged[cols_after_passengers].bfill(axis=1).iloc[:, 0]

df_merged['rate_passengers'] = df_merged['passengers_after'] / df_merged['passengers_before']

In [ ]:
# 人口の増減率
df_merged['rate_population_2km'] = df_merged['population_within_2km_2020'] / df_merged['population_within_2km_2015']
df_merged['rate_population_1km'] = df_merged['population_within_1km_2020'] / df_merged['population_within_1km_2015']
df_merged['rate_population_5km'] = df_merged['population_within_5km_2020'] / df_merged['population_within_5km_2015']
df_merged['rate_population_10km'] = df_merged['population_within_10km_2020'] / df_merged['population_within_10km_2015']

In [ ]:
# 地価の増減率
# land_price_beforeの優先順位リストを定義
cols_before_land_price= [
    'average_land_price_within_2km_2019', 'average_land_price_within_2km_2018',
    'average_land_price_within_2km_2017','average_land_price_within_2km_2016', 'average_land_price_within_2km_2015',
]

# passengers_before列を作成
df_merged['land_price_before_2km'] = df_merged[cols_before_land_price].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_land_price = [
    'average_land_price_within_2km_2024','average_land_price_within_2km_2023', 'average_land_price_within_2km_2022',
    'average_land_price_within_2km_2021', 'average_land_price_within_2km_2020'
]

# passengers_after列を作成
df_merged['land_price_after_2km'] = df_merged[cols_after_land_price].bfill(axis=1).iloc[:, 0]

df_merged['rate_land_price_2km'] = df_merged['land_price_after_2km'] / df_merged['land_price_before_2km']

In [ ]:
# 地価の増減率
# land_price_beforeの優先順位リストを定義
cols_before_land_price= [
    'average_land_price_within_1km_2019', 'average_land_price_within_1km_2018',
    'average_land_price_within_1km_2017','average_land_price_within_1km_2016', 'average_land_price_within_1km_2015',
]

# passengers_before列を作成
df_merged['land_price_before_1km'] = df_merged[cols_before_land_price].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_land_price = [
    'average_land_price_within_1km_2024','average_land_price_within_1km_2023', 'average_land_price_within_1km_2022',
    'average_land_price_within_1km_2021', 'average_land_price_within_1km_2020'
]

# passengers_after列を作成
df_merged['land_price_after_1km'] = df_merged[cols_after_land_price].bfill(axis=1).iloc[:, 0]

df_merged['rate_land_price_1km'] = df_merged['land_price_after_1km'] / df_merged['land_price_before_1km']

In [ ]:
# 地価の増減率
# land_price_beforeの優先順位リストを定義
cols_before_land_price= [
    'average_land_price_within_5km_2019', 'average_land_price_within_5km_2018',
    'average_land_price_within_5km_2017','average_land_price_within_5km_2016', 'average_land_price_within_5km_2015',
]

# passengers_before列を作成
df_merged['land_price_before_5km'] = df_merged[cols_before_land_price].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_land_price = [
    'average_land_price_within_5km_2024','average_land_price_within_5km_2023', 'average_land_price_within_5km_2022',
    'average_land_price_within_5km_2021', 'average_land_price_within_5km_2020'
]

# passengers_after列を作成
df_merged['land_price_after_5km'] = df_merged[cols_after_land_price].bfill(axis=1).iloc[:, 0]

df_merged['rate_land_price_5km'] = df_merged['land_price_after_5km'] / df_merged['land_price_before_5km']

In [ ]:
# 地価の増減率
# land_price_beforeの優先順位リストを定義
cols_before_land_price= [
    'average_land_price_within_10km_2019', 'average_land_price_within_10km_2018',
    'average_land_price_within_10km_2017','average_land_price_within_10km_2016', 'average_land_price_within_10km_2015',
]

# passengers_before列を作成
df_merged['land_price_before_10km'] = df_merged[cols_before_land_price].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_land_price = [
    'average_land_price_within_10km_2024','average_land_price_within_10km_2023', 'average_land_price_within_10km_2022',
    'average_land_price_within_10km_2021', 'average_land_price_within_10km_2020'
]

# passengers_after列を作成
df_merged['land_price_after_10km'] = df_merged[cols_after_land_price].bfill(axis=1).iloc[:, 0]

df_merged['rate_land_price_10km'] = df_merged['land_price_after_10km'] / df_merged['land_price_before_10km']

In [ ]:
# バス停の増減率
df_merged['rate_bus_stops_2km'] = df_merged['bus_stops_within_2km_2022'] / df_merged['bus_stops_within_2km_2010']
df_merged['rate_bus_stops_1km'] = df_merged['bus_stops_within_1km_2022'] / df_merged['bus_stops_within_1km_2010']
df_merged['rate_bus_stops_5km'] = df_merged['bus_stops_within_5km_2022'] / df_merged['bus_stops_within_5km_2010']
df_merged['rate_bus_stops_10km'] = df_merged['bus_stops_within_10km_2022'] / df_merged['bus_stops_within_10km_2010']

In [ ]:
# 事業所数と就業者数の増減率
df_merged['rate_establishments_2km'] = df_merged['establishments_within_2km_2021'] / df_merged['establishments_within_2km_2016']
df_merged['rate_establishments_1km'] = df_merged['establishments_within_1km_2021'] / df_merged['establishments_within_1km_2016']
df_merged['rate_establishments_5km'] = df_merged['establishments_within_5km_2021'] / df_merged['establishments_within_5km_2016']
df_merged['rate_establishments_10km'] = df_merged['establishments_within_10km_2021'] / df_merged['establishments_within_10km_2016']

df_merged['rate_employees_2km'] = df_merged['employees_within_2km_2021'] / df_merged['employees_within_2km_2016']
df_merged['rate_employees_1km'] = df_merged['employees_within_1km_2021'] / df_merged['employees_within_1km_2016']
df_merged['rate_employees_5km'] = df_merged['employees_within_5km_2021'] / df_merged['employees_within_5km_2016']
df_merged['rate_employees_10km'] = df_merged['employees_within_10km_2021'] / df_merged['employees_within_10km_2016']

In [ ]:
# 人口予測の増減率
df_merged['rate_estimated_population_2km'] = df_merged['estimated_population_within_2km_2050'] / df_merged['estimated_population_within_2km_2020']
df_merged['rate_estimated_population_1km'] = df_merged['estimated_population_within_1km_2050'] / df_merged['estimated_population_within_1km_2020']
df_merged['rate_estimated_population_5km'] = df_merged['estimated_population_within_5km_2050'] / df_merged['estimated_population_within_5km_2020']
df_merged['rate_estimated_population_10km'] = df_merged['estimated_population_within_10km_2050'] / df_merged['estimated_population_within_10km_2020']

In [ ]:
df_merged

半径別の増減率

In [ ]:
# 人口
df_merged['population_rate_1km_to_2km_2020'] = df_merged['population_within_2km_2020'] / df_merged['population_within_1km_2020']
df_merged['population_rate_1km_to_5km_2020'] = df_merged['population_within_5km_2020'] / df_merged['population_within_1km_2020']
df_merged['population_rate_1km_to_10km_2020'] = df_merged['population_within_10km_2020'] / df_merged['population_within_1km_2020']
df_merged['population_rate_2km_to_5km_2020'] = df_merged['population_within_5km_2020'] / df_merged['population_within_2km_2020']
df_merged['population_rate_2km_to_10km_2020'] = df_merged['population_within_10km_2020'] / df_merged['population_within_2km_2020']
df_merged['population_rate_5km_to_10km_2020'] = df_merged['population_within_10km_2020'] / df_merged['population_within_5km_2020']

# 地価
df_merged['average_land_price_rate_1km_to_2km_2024'] = df_merged['average_land_price_within_2km_2024'] / df_merged['average_land_price_within_1km_2024']
df_merged['average_land_price_rate_1km_to_5km_2024'] = df_merged['average_land_price_within_5km_2024'] / df_merged['average_land_price_within_1km_2024']
df_merged['average_land_price_rate_1km_to_10km_2024'] = df_merged['average_land_price_within_10km_2024'] / df_merged['average_land_price_within_1km_2024']
df_merged['average_land_price_rate_2km_to_5km_2024'] = df_merged['average_land_price_within_5km_2024'] / df_merged['average_land_price_within_2km_2024']
df_merged['average_land_price_rate_2km_to_10km_2024'] = df_merged['average_land_price_within_10km_2024'] / df_merged['average_land_price_within_2km_2024']
df_merged['average_land_price_rate_5km_to_10km_2024'] = df_merged['average_land_price_within_10km_2024'] / df_merged['average_land_price_within_5km_2024']

# バス停
df_merged['bus_stops_rate_1km_to_2km_2022'] = df_merged['bus_stops_within_2km_2022'] / df_merged['bus_stops_within_1km_2022']
df_merged['bus_stops_rate_1km_to_5km_2022'] = df_merged['bus_stops_within_5km_2022'] / df_merged['bus_stops_within_1km_2022']
df_merged['bus_stops_rate_1km_to_10km_2022'] = df_merged['bus_stops_within_10km_2022'] / df_merged['bus_stops_within_1km_2022']
df_merged['bus_stops_rate_2km_to_5km_2022'] = df_merged['bus_stops_within_5km_2022'] / df_merged['bus_stops_within_2km_2022']
df_merged['bus_stops_rate_2km_to_10km_2022'] = df_merged['bus_stops_within_10km_2022'] / df_merged['bus_stops_within_2km_2022']
df_merged['bus_stops_rate_5km_to_10km_2022'] = df_merged['bus_stops_within_10km_2022'] / df_merged['bus_stops_within_5km_2022']

# 事業所数
df_merged['establishments_rate_1km_to_2km_2021'] = df_merged['establishments_within_2km_2021'] / df_merged['establishments_within_1km_2021']
df_merged['establishments_rate_1km_to_5km_2021'] = df_merged['establishments_within_5km_2021'] / df_merged['establishments_within_1km_2021']
df_merged['establishments_rate_1km_to_10km_2021'] = df_merged['establishments_within_10km_2021'] / df_merged['establishments_within_1km_2021']
df_merged['establishments_rate_2km_to_5km_2021'] = df_merged['establishments_within_5km_2021'] / df_merged['establishments_within_2km_2021']
df_merged['establishments_rate_2km_to_10km_2021'] = df_merged['establishments_within_10km_2021'] / df_merged['establishments_within_2km_2021']
df_merged['establishments_rate_5km_to_10km_2021'] = df_merged['establishments_within_10km_2021'] / df_merged['establishments_within_5km_2021']

# 就業者数
df_merged['employees_rate_1km_to_2km_2021'] = df_merged['employees_within_2km_2021'] / df_merged['employees_within_1km_2021']
df_merged['employees_rate_1km_to_5km_2021'] = df_merged['employees_within_5km_2021'] / df_merged['employees_within_1km_2021']
df_merged['employees_rate_1km_to_10km_2021'] = df_merged['employees_within_10km_2021'] / df_merged['employees_within_1km_2021']
df_merged['employees_rate_2km_to_5km_2021'] = df_merged['employees_within_5km_2021'] / df_merged['employees_within_2km_2021']
df_merged['employees_rate_2km_to_10km_2021'] = df_merged['employees_within_10km_2021'] / df_merged['employees_within_2km_2021']
df_merged['employees_rate_5km_to_10km_2021'] = df_merged['employees_within_10km_2021'] / df_merged['employees_within_5km_2021']

# 将来人口
df_merged['estimated_population_rate_1km_to_2km_2050'] = df_merged['estimated_population_within_2km_2050'] / df_merged['estimated_population_within_1km_2050']
df_merged['estimated_population_rate_1km_to_5km_2050'] = df_merged['estimated_population_within_5km_2050'] / df_merged['estimated_population_within_1km_2050']
df_merged['estimated_population_rate_1km_to_10km_2050'] = df_merged['estimated_population_within_10km_2050'] / df_merged['estimated_population_within_1km_2050']
df_merged['estimated_population_rate_2km_to_5km_2050'] = df_merged['estimated_population_within_5km_2050'] / df_merged['estimated_population_within_2km_2050']
df_merged['estimated_population_rate_2km_to_10km_2050'] = df_merged['estimated_population_within_10km_2050'] / df_merged['estimated_population_within_2km_2050']
df_merged['estimated_population_rate_5km_to_10km_2050'] = df_merged['estimated_population_within_10km_2050'] / df_merged['estimated_population_within_5km_2050']

df_merged

（１０）データの保存

In [ ]:
# 'column_name' の列に "Tokyo" を含む行を抽出
df_tokyo = df_merged[df_merged['jan_station'].str.contains('東京', case=False, na=False)]
df_tokyo

In [ ]:
# 削除したい列をリストで指定して削除
columns_to_drop = ['sameAs','railway','station','operator','includeAlighting',
                   'trans_sameAs','connectingRailway_number','connectingStation_number',
                   'passengers_before','passengers_after',
                   'land_price_before_2km','land_price_before_1km','land_price_before_5km','land_price_before_10km',
                   'land_price_after_2km','land_price_after_1km','land_price_after_5km','land_price_after_10km','prefecture']
df_merged = df_merged.drop(columns=columns_to_drop).reset_index(drop=True)

In [ ]:
df_merged['station_id'] = df_merged.index
df_merged

In [ ]:
# 変換したい列名をリストにまとめる
columns_to_convert = [
    'passengers_2000', 'passengers_2001', 'passengers_2002', 'passengers_2003', 'passengers_2004',
    'passengers_2005', 'passengers_2006', 'passengers_2007', 'passengers_2008', 'passengers_2009',
    'passengers_2010', 'passengers_2011', 'passengers_2012', 'passengers_2013', 'passengers_2014',
    'passengers_2015', 'passengers_2016', 'passengers_2017', 'passengers_2018', 'passengers_2019',
    'passengers_2020', 'passengers_2021', 'passengers_2022', 'passengers_2023',
    'connect_line_number', 'connect_company_number',
    'bus_stops_within_2km_2022', 'bus_stops_within_2km_2010',
    'bus_stops_within_1km_2022', 'bus_stops_within_1km_2010',
    'bus_stops_within_5km_2022', 'bus_stops_within_5km_2010',
    'bus_stops_within_10km_2022', 'bus_stops_within_10km_2010'
]

# 指定した列を整数型に変換
for column in columns_to_convert:
    df_merged[column] = pd.to_numeric(df_merged[column], errors='coerce').astype('Int64')  # 欠損値を保持するためにInt64型を使用

df_merged

In [ ]:
# 条件に基づいて rate_passengers を <NA> に設定
df_merged.loc[(df_merged['jan_station'] == '越生') & (df_merged['pref_name'] == '埼玉県'), 'rate_passengers'] = pd.NA

df_merged.loc[(df_merged['jan_station'] == '新整備場') & (df_merged['pref_name'] == '東京都'),
 ['rate_population_2km', 'rate_population_1km','rate_population_5km','rate_population_10km']] = pd.NA

In [ ]:
print(df_merged['connect_line_number'].dtypes)

In [ ]:
nan_counts = df_merged.isna().sum()

# 表示の上限を一時的に変更
with pd.option_context('display.max_rows', 70):
    print(nan_counts.head(70))

In [ ]:
zero_counts = (df_merged == 0).sum()
# 表示の上限を一時的に変更
with pd.option_context('display.max_rows', 70):
    print(zero_counts.head(70))

In [ ]:
df_merged

In [ ]:
# cp932エンコーディングでCSVファイルとして保存
df_merged.to_csv('station_are_dataset_cp932.csv', encoding='cp932',errors='replace', index=False)

In [ ]:
df_merged.to_csv('station_are_dataset_utf-8.csv', encoding='utf-8', index=False)

In [ ]:
df_merged.to_csv('station_area_dataset_shift-jis.csv', encoding='shift-jis', index=False, errors='replace')

（１０）乗降客数の予測

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 削除する特徴量を指定
features_to_drop = ['station_id','sameAs','railway','station','includeAlighting','operator',
                    'passengers_2023','trans_sameAs','jan_station','connectingStation_number',
                    'geometry','estimated_population_within_2km_2020','estimated_population_within_2km_2025',
                    'estimated_population_within_2km_2030','estimated_population_within_2km_2035',
                    'estimated_population_within_2km_2040','estimated_population_within_2km_2045',
                    'estimated_population_within_2km_2050',
                    'connectingRailway_number','connect_line_name','connect_company_name']

# 特徴量を削除
df_predict = df_merged.drop(columns=features_to_drop)

# 目的変数のpassengers_2022がNANの行を削除
df_predict = df_predict.dropna(subset=['passengers_2022'])

# ワンホットエンコーディングを適用
# df_predict= pd.get_dummies(df_predict, columns=['operator'])

# 目的変数 (例: 'target') と特徴量に分ける
X = df_predict.drop(columns=['passengers_2022'])
y = df_predict['passengers_2022']

# 学習データとテストデータに8:2で分割（ランダムステート42を指定）
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 学習データをさらに80%:20%に分割して、バリデーションデータを作成
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# LightGBMのデータセット形式に変換
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid)

# モデルのパラメータ設定
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'num_leaves': 1000,
    'min_data_in_leaf': 20,
    "n_estimators": 20000,
    "colsample_bytree":1.0,
    "subsample":1.0,
    "subsample_freq":1,
    "max_depth":-1,
    'verbose': -1
}

# early_stopping用の結果を保持する辞書
evals_result = {}

# モデルの学習
gbm = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, valid_data],  # valid_setsに学習データとバリデーションデータを指定
    valid_names=['train', 'valid'],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(stopping_rounds=100)]  # early_stopping_roundsの代わりにcallbacksを使用
)

# テストデータで予測
y_pred = gbm.predict(X_test, num_iteration=gbm.best_iteration)

# # RMSEを計算
# rmse = mean_squared_error(y_test, y_pred, squared=False)
# print(f'RMSE: {rmse}')

# 予測誤差を計算 (予測値 / 実績値)
prediction_error = y_pred / y_test

# 予測誤差が±20%の件数をカウント
error_within_20_percent = ((prediction_error >= 0.8) & (prediction_error <= 1.2)).sum()

# 全体の件数
total_count = len(y_pred)

# 予測誤差が±20%の割合を計算
percentage_within_20_percent = (error_within_20_percent / total_count) * 100

print(f'予測誤差が±20%の割合: {percentage_within_20_percent:.2f}%')

（１１）増減率のデータフレームを作成

In [ ]:
df_rate = df_merged.copy()

In [ ]:
# 乗降者数の増減率
# passengers_beforeの優先順位リストを定義
cols_before_passengers = [
    'passengers_2019', 'passengers_2018', 'passengers_2017',
    'passengers_2016', 'passengers_2015', 'passengers_2014',
    'passengers_2013', 'passengers_2012', 'passengers_2011'
]

# passengers_before列を作成
df_rate['passengers_before'] = df_rate[cols_before_passengers].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_passengers = [
    'passengers_2023', 'passengers_2022',
    'passengers_2021', 'passengers_2020'
]

# passengers_after列を作成
df_rate['passengers_after'] = df_rate[cols_after_passengers].bfill(axis=1).iloc[:, 0]

df_rate['rate_passengers'] = df_rate['passengers_after'] / df_rate['passengers_before']

In [ ]:
# 人口の増減率
df_rate['rate_population'] = df_rate['population_within_2km_2020'] / df_rate['population_within_2km_2015']

In [ ]:
# 乗降者数の増減率
# land_price_beforeの優先順位リストを定義
cols_before_land_price= [
    'average_land_price_within_2km_2019', 'average_land_price_within_2km_2018',
    'average_land_price_within_2km_2017','average_land_price_within_2km_2016', 'average_land_price_within_2km_2015',
]

# passengers_before列を作成
df_rate['land_price_before'] = df_rate[cols_before_land_price].bfill(axis=1).iloc[:, 0]

# passengers_afterの優先順位リストを定義
cols_after_land_price = [
    'average_land_price_within_2km_2024','average_land_price_within_2km_2023', 'average_land_price_within_2km_2022',
    'average_land_price_within_2km_2021', 'average_land_price_within_2km_2020'
]

# passengers_after列を作成
df_rate['land_price_after'] = df_rate[cols_after_land_price].bfill(axis=1).iloc[:, 0]

df_rate['rate_land_price'] = df_rate['land_price_after'] / df_rate['land_price_before']

In [ ]:
# バス停の増減率
df_rate['rate_bus_stops'] = df_rate['bus_stops_within_2km_2022_2023'] / df_rate['bus_stops_within_2km_2010']

In [ ]:
# 事業所数と就業者数の増減率
df_rate['rate_establishments'] = df_rate['establishments_within_2km_2021'] / df_rate['establishments_within_2km_2016']
df_rate['employees'] = df_rate['employees_within_2km_2021'] / df_rate['employees_within_2km_2016']

In [ ]:
df_rate

In [ ]:
# 残すカラムのリストを定義
cols_to_keep = [
    'trans_sameAs',
    'jan_station',
    'connectingRailway_number',
    'latitude',
    'longitude',
    'rate_passengers',
    'rate_population',
    'rate_land_price',
    'rate_bus_stops',
    'rate_establishments',
    'employees'
]

# 指定したカラムのみを残す
df_rate = df_rate[cols_to_keep]

In [ ]:
nan_counts = df_rate.isna().sum()

print(nan_counts)

In [ ]:
# 'rate_passengers'がNaNの行を削除
df_rate = df_rate.dropna(subset=['rate_passengers']).reset_index(drop=True)

df_rate

参考

In [ ]:
import requests
import json

# アクセストークン
access_token = 'hid49u64myobsb2buk6pcqdlbq6baut8slmdpx7vjecemq15dhrzao8qwr3brwg9'

# リクエストのエンドポイント
# url = 'https://api-challenge2024.odpt.org/api/v4/odpt:PassengerSurvey'
url = 'https://api-challenge2024.odpt.org/api/v4/odpt:Station'
# url = 'https://api-challenge2024.odpt.org/api/v4/odpt:TrainInformation'

# クエリパラメータ
params = {
    'odpt:operator': 'odpt.Operator:JR-East',
    # 'odpt:operator': 'odpt.Operator:jre-is',

    'acl:consumerKey': access_token
}

# リクエストを送信
response = requests.get(url, params=params)

# レスポンスを表示
if response.status_code == 200:
    data = response.json()  # JSON形式でレスポンスを取得
    # 見やすく整形して表示
    print(json.dumps(data, indent=4, ensure_ascii=False))
else:
    print(f"Error: {response.status_code}")

In [ ]:
import requests

# アクセストークン
access_token = 'hid49u64myobsb2buk6pcqdlbq6baut8slmdpx7vjecemq15dhrzao8qwr3brwg9'

# リクエストのエンドポイント
url = 'https://api-challenge2024.odpt.org/api/v4/gtfs/realtime/jreast_odpt_train_vehicle'

# クエリパラメータ
params = {
    'acl:consumerKey': access_token
}

# リクエストを送信
response = requests.get(url, params=params)

# レスポンスを表示
if response.status_code == 200:
    # バイナリ形式でのレスポンスを取得
    gtfs_rt_data = response.content
    # 取得したバイナリデータをファイルに保存する（オプション）
    with open('jreast_train_realtime.pb', 'wb') as file:
        file.write(gtfs_rt_data)
    print("GTFS-RT data has been successfully retrieved and saved.")
else:
    print(f"Error: {response.status_code}")


In [ ]:
!pip install gtfs-realtime-bindings
import requests
from google.transit import gtfs_realtime_pb2

# アクセストークン
access_token = 'hid49u64myobsb2buk6pcqdlbq6baut8slmdpx7vjecemq15dhrzao8qwr3brwg9'

# リクエストのエンドポイント
url = 'https://api-challenge2024.odpt.org/api/v4/gtfs/realtime/jreast_odpt_train_vehicle'

# クエリパラメータ
params = {
    'acl:consumerKey': access_token
}

# リクエストを送信
response = requests.get(url, params=params)

# レスポンスを表示
if response.status_code == 200:
    # バイナリ形式でのレスポンスを取得
    gtfs_rt_data = response.content

    # GTFS-RTデータをデコード
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(gtfs_rt_data)

    # デコードされたデータを表示
    for entity in feed.entity:
        if entity.HasField('vehicle'):
            print(f"ID: {entity.id}")
            print(f"Vehicle ID: {entity.vehicle.vehicle.id}")
            print(f"Position: lat={entity.vehicle.position.latitude}, lon={entity.vehicle.position.longitude}")
            print(f"Current Stop ID: {entity.vehicle.stop_id}")
            print("-----------")
else:
    print(f"Error: {response.status_code}")